# Evaporating Universe — Paper I
## NB04 — Observational Validation
## NB03: Observational Validation

**Google Colab Notebook** — Compares CLASS-EU outputs against
observational data from Planck 2018, DESI DR1, Pantheon+, SH0ES, and KiDS/DES.

This notebook:
1. Imports CLASS outputs from NB02 (`NB03_A_results.json + NB02_results.json` + `.dat` files)
2. Extracts $\sigma_8$ and $r_d$ from CLASS output
3. Compares CMB $C_\ell$ against Planck 2018 bandpowers
4. Validates full-shape clustering against DESI DR1 ShapeFit (4×4 covariance)
5. Quantifies $S_8$ resolution vs KiDS/DES
6. Decomposes $H_0$ via GKI/LKI channels
7. Checks ISW enhancement and CMB lensing
8. Model selection: $\Delta\chi^2$, $\Delta$AIC, $\Delta$BIC


> **Modes:** This notebook supports two parameter modes:
> - **Mode A** (default): Planck ΛCDM fiducial parameters → CLASS-EU (demonstration)
> - **Mode B**: MCMC C2 posterior parameters → CLASS-EU (paper-ready, self-consistent)
>
> ΛCDM reference curves are **identical** in both modes. Only the EU side changes.
> **Auto-detect**: Mode B runs automatically if `NB05_C2_results.json` is uploaded.


---
# §1. Setup

Import parameters from NB01 and CLASS output from NB02.
Upload a `.zip` with all required files, or upload individually.
Upload required files via zip.


In [ ]:
# ============================================================
# §1. SETUP — Import NB01 + NB02 results
# ============================================================
# Upload a .zip with all required files
# ============================================================

import os, json, shutil, glob, zipfile
import numpy as np
from scipy.integrate import quad
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import re
import urllib.request

DATA_DIR = '/content/nb02_output'
os.makedirs(DATA_DIR, exist_ok=True)

required_dat = ['lcdm_cl.dat', 'eu_cl.dat', 'lcdm_pk.dat', 'eu_pk.dat',
                'lcdm_background.dat', 'eu_background.dat',
                'lcdm_cl_lensed.dat', 'eu_cl_lensed.dat']
required_json = ['NB01_params.json', 'NB02_results.json', 'NB03_A_results.json']

def check_files():
    """Check if all required files are present."""
    for f in required_json:
        if not os.path.exists(os.path.join('/content', f)):
            return False
    for f in required_dat:
        if not os.path.exists(os.path.join(DATA_DIR, f)):
            return False
    return True

# ── Strategy 1: Check if files already present (re-run) ──
if check_files():
    print('[OK] All files already present (re-run detected)')

else:
    # ── Strategy 2: Upload ──
    print('Upload NB03_outputs.zip (recommended) or individual files:')
    print()
    print('  JSONs (3+1):')
    print('    1. NB01_params.json')
    print('    2. NB02_results.json')
    print('    3. NB03_A_results.json')
    print('    3b. NB03_B_results.json  ← Mode B (if available)')
    print('    4. NB05_C2_results.json  ← Mode B (paper-ready)')
    print()
    print('  CLASS outputs — Mode A (8):')
    print('    5. lcdm_background.dat')
    print('    6. eu_background.dat')
    print('    7. lcdm_cl.dat')
    print('    8. eu_cl.dat')
    print('    9. lcdm_cl_lensed.dat')
    print('   10. eu_cl_lensed.dat')
    print('   11. lcdm_pk.dat')
    print('   12. eu_pk.dat')
    print()
    print('  CLASS outputs — Mode B (4, from NB03 §7):')
    print('   13. eu_modeB_background.dat')
    print('   14. eu_modeB_cl.dat')
    print('   15. eu_modeB_cl_lensed.dat')
    print('   16. eu_modeB_pk.dat')
    
    from google.colab import files
    uploaded = files.upload()
    
    for name in uploaded:
        src_path = os.path.join('/content', name)
        
        # Handle .zip upload
        if name.endswith('.zip'):
            print(f'  Extracting {name}...')
            with zipfile.ZipFile(src_path, 'r') as zf:
                for member in zf.namelist():
                    basename = os.path.basename(member)
                    if not basename:
                        continue
                    data = zf.read(member)
                    if basename.endswith('.dat'):
                        dst = os.path.join(DATA_DIR, basename)
                        with open(dst, 'wb') as f:
                            f.write(data)
                        print(f'    {basename} -> {DATA_DIR}/')
                    elif basename.endswith('.json'):
                        dst = os.path.join('/content', basename)
                        with open(dst, 'wb') as f:
                            f.write(data)
                        print(f'    {basename} -> /content/')
            os.remove(src_path)
        
        # Handle individual files
        elif name.endswith('.dat'):
            dst = os.path.join(DATA_DIR, name)
            if src_path != dst:
                shutil.move(src_path, dst)
            print(f'  {name} -> {DATA_DIR}/')
        elif name.endswith('.json'):
            print(f'  {name} -> /content/')

# ── Verify all files ──
assert os.path.exists('/content/NB01_params.json'), 'FATAL: NB01_params.json missing'
# NB02/NB03 JSON assertions handled by the load block above
for f in required_dat:
    assert os.path.exists(os.path.join(DATA_DIR, f)), f'FATAL: {f} missing from {DATA_DIR}'
print('[OK] All files verified')

# ── Load parameters ──
with open('/content/NB01_params.json', 'r') as f:
    params = json.load(f)
with open('/content/NB02_results.json', 'r') as f:
    nb02 = json.load(f)
with open('/content/NB03_A_results.json', 'r') as f:
    nb03 = json.load(f)

H0_Planck = params['planck2018_LCDM_derived']['H0']
Omega_m   = params['planck2018_LCDM_derived']['Omega_m']
Omega_b   = params['planck2018_LCDM_derived']['Omega_b_LCDM']
Omega_c   = params['planck2018_LCDM_derived']['Omega_c']
Omega_L   = params['planck2018_LCDM_derived']['Omega_L']
sigma8_P  = params['planck2018_LCDM_derived']['sigma8']
H0_SH0ES     = params['shoes2024']['H0']
H0_SH0ES_err = params['shoes2024']['H0_err']
b       = params['eu_derived']['b']['value']
z_trans = params['eu_derived']['z_trans']['value']
eps_IR  = params['eu_derived']['eps_IR']['value']
lam     = params['eu_derived']['lambda']['value']

H0_EU = nb02['lki_void']['H0_local']  # GKI + KBC Void (universal)

def epsilon_of_z(z):
    x = ((1 + z) / (1 + z_trans))**(1.0/b)
    return eps_IR / (1.0 + x)

print(f'  H0(Planck) = {H0_Planck}, H0(EU) = {H0_EU:.2f}, H0(SH0ES) = {H0_SH0ES}')
# ── MODE NOTE ──
# H0_Planck, Omega_m, Omega_b, Omega_c, Omega_L, sigma8_P:
#   These are ΛCDM reference values. They are IDENTICAL in Mode A and B.
#   Mode B only changes the EU CLASS run, not the ΛCDM baseline.
# H0_EU (from NB02): UV analytical prediction. IDENTICAL in both modes.


In [ ]:
# ============================================================
# §1.5 MODE SWITCH — Select parameter source
# ============================================================
# Mode A: Planck ΛCDM fiducial → CLASS-EU (demonstration)
# Mode B: MCMC C2 posteriors → CLASS-EU (paper-ready)
#
# ΛCDM reference is IDENTICAL in both modes.
# EU theory params (ε, z_t, b) are IDENTICAL in both modes.
# Only the 6 base cosmo params for CLASS-EU change.
# ============================================================

# ============================================================
# AUTO-DETECT: Check if C2 results exist
# Mode A always runs. Mode B runs automatically if C2 exists.
# ============================================================
import os as _os
_c2_paths = ['/content/NB05_C2_results.json', 'NB05_C2_results.json',
             'results/NB05_C2_results.json']
_c2_found = any(_os.path.exists(p) for p in _c2_paths)

# FIRST PASS: Always Mode A
USE_MCMC_PARAMS = False
DUAL_MODE = _c2_found  # Will trigger Mode B after Mode A completes

if DUAL_MODE:
    print('[AUTO-DETECT] NB05_C2_results.json FOUND')
    print('  \u2192 Will run Mode A first, then Mode B automatically')
    print('  \u2192 Exports: NB04_A_results.json + NB04_B_results.json')
else:
    print('[AUTO-DETECT] NB05_C2_results.json not found')
    print('  \u2192 Running Mode A only')
    print('  \u2192 Export: NB04_A_results.json')

if USE_MCMC_PARAMS:
    # Load C2 MCMC results
    _c2_path = '/content/NB05_C2_results.json'
    if not os.path.exists(_c2_path):
        # Try results/ directory
        _c2_path = 'NB05_C2_results.json'
    assert os.path.exists(_c2_path), (
        f'FATAL: NB05_C2_results.json not found. '
        f'Upload it alongside the other JSONs for Mode B.'
    )
    with open(_c2_path, 'r') as _f:
        c2 = json.load(_f)
    
    MODE = 'B'
    MODE_LABEL = 'MCMC C2 (paper-ready)'
    
    # C2 derived quantities (for cross-checks and Bridge §2)
    _cp = c2['cosmological_params']
    _dp = c2['derived_params']
    C2_COSMO = {
        'omega_b':     _cp['omega_b']['mean'],
        'omega_cdm':   _cp['omega_cdm']['mean'],
        'theta_s_100': _cp['theta_s_100']['mean'],
        'tau_reio':    _cp['tau_reio']['mean'],
        'logA':        _cp['logA']['mean'],
        'n_s':         _cp['n_s']['mean'],
    }
    C2_DERIVED = {
        'H0':       _dp['H0']['mean'],
        'sigma8':   _dp['sigma8']['mean'],
        'S8':       _dp['S8']['mean'],
        'Omega_m':  _dp['Omega_m']['mean'],
        'fcdm_z0':  _dp['fcdm_z0']['mean'],
        'H0_LKI':   _dp['H0_LKI']['mean'],
        'rdrag':    _dp['rdrag']['mean'],  # MUST exist — no fallback
    }
    
    # Validate all required keys exist (FAIL-FAST)
    _required = ['H0', 'sigma8', 'S8', 'Omega_m', 'fcdm_z0', 'H0_LKI', 'rdrag']
    _missing = [k for k in _required if k not in C2_DERIVED]
    assert not _missing, (
        f'FATAL: NB05_C2_results.json missing derived_params: {_missing}. '
        f'Re-export from analysis_RUN_C2.json with all fields.'
    )
    
    print(f'[MODE B] EU CLASS will use MCMC C2 parameters')
    print(f'  Source: {_c2_path} (R-1 = {c2["_metadata"]["convergence"]})')
    print(f'  H0_GKI expected: ~{C2_DERIVED["H0"]:.3f} km/s/Mpc')
    print(f'  σ8 expected:     ~{C2_DERIVED["sigma8"]:.4f}')
    print(f'  S8 expected:     ~{C2_DERIVED["S8"]:.4f}')
else:
    c2 = None
    C2_COSMO = None
    C2_DERIVED = None
    MODE = 'A'
    MODE_LABEL = 'Planck fiducial (demonstration)'
    print(f'[MODE A] EU CLASS will use Planck ΛCDM fiducial parameters')
    print(f'  Source: NB01_params.json → NB03 .dat files')

print(f'  ΛCDM reference: UNCHANGED in both modes (H0=67.36)')


In [ ]:
# ============================================================
# §1.7 CLASS RE-RUN — Mode B only
# ============================================================
# If Mode B, re-generate EU .dat files using C2 MCMC parameters.
# Two strategies:
#   1. If pre-computed Mode B .dat files exist → use them
#   2. Otherwise → run CLASS-EU with C2 params (~30s)
#
# ΛCDM .dat files are NEVER touched.
# EU theory params (ε, z_t, b) are IDENTICAL to Mode A.
# ============================================================

if USE_MCMC_PARAMS:
    # Strategy 1: check for pre-computed Mode B files
    _modeB_dat = ['eu_modeB_background.dat', 'eu_modeB_cl.dat', 
                  'eu_modeB_pk.dat', 'eu_modeB_cl_lensed.dat']
    _has_precomputed = all(
        os.path.exists(os.path.join(DATA_DIR, f)) for f in _modeB_dat
    )
    
    if _has_precomputed:
        print('[MODE B] Using pre-computed Mode B .dat files')
        # Overwrite EU files with Mode B versions
        for f_b in _modeB_dat:
            f_a = f_b.replace('eu_modeB_', 'eu_')
            src = os.path.join(DATA_DIR, f_b)
            dst = os.path.join(DATA_DIR, f_a)
            shutil.copy2(src, dst)
            print(f'  {f_b} → {f_a}')
        CLASS_EU_RERUN = False
    else:
        # Strategy 2: run CLASS-EU with C2 params
        print('[MODE B] No pre-computed files found. Running CLASS-EU...')
        try:
            from classy import Class
            
            eu_cfg = {
                'eu_epsilon_ir': params['eu_derived']['eps_IR']['value'],
                'eu_z_trans':    params['eu_derived']['z_trans']['value'],
                'eu_b':          params['eu_derived']['b']['value'],
                'eu_lambda':     params['eu_derived']['lambda']['value'],
                'ide_perturbations': 1,
            }
            base_cfg = {
                'omega_b':        C2_COSMO['omega_b'],
                'omega_cdm':      C2_COSMO['omega_cdm'],
                '100*theta_s':    C2_COSMO['theta_s_100'],
                'tau_reio':       C2_COSMO['tau_reio'],
                'ln10^{10}A_s':   C2_COSMO['logA'],
                'n_s':            C2_COSMO['n_s'],
            }
            class_cfg = {
                'output': 'tCl,pCl,lCl,mPk',
                'lensing': 'yes',
                'P_k_max_1/Mpc': 10.0,
                'l_max_scalars': 2600,
                'm_ncdm': 0.0589,
                'N_ur': 2.0328,
                'N_ncdm': 1,
            }
            cosmo_eu_B = Class()
            cosmo_eu_B.set({**base_cfg, **eu_cfg, **class_cfg})
            cosmo_eu_B.compute()
            
            H0_check = cosmo_eu_B.h() * 100
            print(f'  CLASS-EU(C2): H0 = {H0_check:.3f} km/s/Mpc')
            print(f'  Expected:     H0 ≈ {C2_DERIVED["H0"]:.3f}')
            assert abs(H0_check - C2_DERIVED['H0']) < 0.5, \
                f'H0 mismatch: CLASS={H0_check:.3f} vs C2={C2_DERIVED["H0"]:.3f}'
            
            # TODO: Export .dat files from cosmo_eu_B object
            # For now, this branch is a placeholder.
            # Recommended: pre-compute .dat files locally.
            CLASS_EU_RERUN = True
            print('  [OK] CLASS-EU re-run successful')
            
        except ImportError:
            print('  [WARN] classy not available. Please use pre-computed .dat files.')
            print('  Upload eu_modeB_background.dat, eu_modeB_cl.dat, etc.')
            CLASS_EU_RERUN = False
else:
    CLASS_EU_RERUN = False
    print('[MODE A] Using original NB03 .dat files (no re-run)')


---
---
# Part II — CMB Safety

**Critical test**: does the EU coupling break the CMB?
We demonstrate that $\Delta C_\ell / C_\ell < 0.1\%$ for $\ell > 30$,
with modifications confined to the cosmic-variance-dominated ISW regime.


---
# §2. Bridge: Extract $\sigma_8$ and $r_d$ from CLASS

The sound horizon $r_d$ and $\sigma_8$ are key derived quantities
that CLASS computes internally. We extract them from the output files.


In [ ]:
# ============================================================
# §2. BRIDGE — Extract sigma8, r_d from CLASS output
# ============================================================

# ── Parse CLASS background for r_d (sound horizon at drag) ──
def load_class_bg(path):
    with open(path, 'r') as f:
        header_line = ''
        for line in f:
            if line.startswith('#'):
                header_line = line
            else:
                break
    parts = re.split(r'(?<=\s)(\d+):', header_line)
    col_map = {}
    for i in range(1, len(parts)-1, 2):
        col_num = int(parts[i]) - 1
        col_name = parts[i+1].strip()
        col_map[col_name] = col_num
    data = np.loadtxt(path)
    return data, col_map

bg_lcdm, cols_l = load_class_bg(os.path.join(DATA_DIR, 'lcdm_background.dat'))
bg_eu, cols_e   = load_class_bg(os.path.join(DATA_DIR, 'eu_background.dat'))

z_l = bg_lcdm[:, cols_l['z']]; H_l = bg_lcdm[:, cols_l['H [1/Mpc]']]
z_e = bg_eu[:, cols_e['z']];   H_e = bg_eu[:, cols_e['H [1/Mpc]']]

# Sound horizon: r_d = comoving sound horizon at z_drag
# CLASS outputs this in the thermodynamics file or we compute from background
# r_d ~ comov.snd.hrz. at z_drag ~ 1060
# Import r_d from NB03 JSON (exact CLASS value, not interpolated)
# r_d: Mode B uses MCMC C2 value, Mode A uses NB03
if USE_MCMC_PARAMS:
    rd_eu = C2_DERIVED['rdrag']  # 147.50 Mpc (from MCMC C2)
    print(f'  [MODE B] r_d(EU) = {rd_eu:.2f} Mpc (from MCMC C2)')
else:
    rd_eu = nb03['CLASS_background']['r_d_EU_Mpc']
rd_lcdm = nb03['CLASS_background']['r_d_LCDM_Mpc']
assert rd_eu > 140 and rd_eu < 150, f"r_d_EU sanity check failed: {rd_eu}"
assert rd_lcdm > 140 and rd_lcdm < 150, f"r_d_LCDM sanity check failed: {rd_lcdm}"

# ── Extract h = H0/100 for each model from CLASS background ──
c_kms = 299792.458  # speed of light in km/s
idx_z0_l = np.argmin(np.abs(z_l))
idx_z0_e = np.argmin(np.abs(z_e))
h_lcdm = (H_l[idx_z0_l] * c_kms) / 100.0  # H(z=0) in km/s/Mpc / 100
h_eu   = (H_e[idx_z0_e] * c_kms) / 100.0
print(f'  h(LCDM) = {h_lcdm:.4f},  h(EU) = {h_eu:.4f}')

# ── sigma8 from P(k) ──
# sigma8^2 = integral of P(k) * W^2(kR) * k^2 dk / (2pi^2)
# IMPORTANT: R = 8 h^{-1} Mpc (standard cosmological definition)
# CLASS P(k) output is in h/Mpc (k) and (Mpc/h)^3 (P).
# In these units, R = 8 Mpc/h (dimensionless: k*R -> h/Mpc * Mpc/h = 1).
# No conversion by h is needed.
def compute_sigma8(pk_file):
    pk = np.loadtxt(pk_file)
    k = pk[:, 0]   # h/Mpc (CLASS default output units)
    Pk = pk[:, 1]  # (Mpc/h)^3
    R = 8.0        # 8 Mpc/h — same unit system as 1/k
    x = k * R
    W = 3.0 * (np.sin(x) - x * np.cos(x)) / x**3
    integrand = Pk * W**2 * k**2 / (2 * np.pi**2)
    sigma8_sq = np.trapezoid(integrand, k)
    return np.sqrt(sigma8_sq)

# sigma8 from P(k) integration (cross-check)
sigma8_lcdm_local = compute_sigma8(os.path.join(DATA_DIR, 'lcdm_pk.dat'))
sigma8_eu_local   = compute_sigma8(os.path.join(DATA_DIR, 'eu_pk.dat'))

# Official values from NB03 JSON (Growth ODE — h-artifact free)
sigma8_lcdm = nb03['sigma8']['sigma8_lcdm']  # ΛCDM: ALWAYS from NB03

if USE_MCMC_PARAMS:
    # Mode B: EU values from C2 MCMC posteriors (data-driven)
    sigma8_eu = C2_DERIVED['sigma8']   # 0.8274 (vs 0.8076 in Mode A)
    print(f'  [MODE B] sigma8_eu = {sigma8_eu:.4f} (from MCMC C2)')
else:
    sigma8_eu = nb03['sigma8']['sigma8_eu']  # Mode A: from NB03

# Cross-check: local P(k) integration vs JSON should be close
assert abs(sigma8_lcdm_local - sigma8_lcdm) < 0.01, \
    f"sigma8_lcdm drift: local={sigma8_lcdm_local:.4f} vs JSON={sigma8_lcdm:.4f}"
print(f'  sigma8 cross-check: P(k)={sigma8_lcdm_local:.4f} vs JSON={sigma8_lcdm:.4f} [OK]')

# Omega_m at z=0: LCDM uses input, EU uses CLASS background (drained by evaporation)
Omega_m_lcdm = Omega_m  # Planck input = 0.3153
# EU: Import from NB03 JSON (includes neutrinos, validated)
if USE_MCMC_PARAMS:
    Omega_m_eu = C2_DERIVED['Omega_m']      # 0.2883 (vs 0.2929 in Mode A)
    f_cdm_survived = C2_DERIVED['fcdm_z0']  # 0.95577 (identical)
else:
    Omega_m_eu = nb03['CLASS_background']['Omega_m_z0_incl_nu']
    f_cdm_survived = nb03['bianchi']['fcdm_z0']
assert 0.25 < Omega_m_eu < 0.35, f"Omega_m_eu sanity check failed: {Omega_m_eu}"
assert 0.90 < f_cdm_survived < 1.0, f"f_cdm sanity check failed: {f_cdm_survived}"

S8_lcdm = nb03['sigma8']['S8_lcdm']  # ΛCDM: ALWAYS from NB03

if USE_MCMC_PARAMS:
    S8_eu = C2_DERIVED['S8']           # 0.8112 (vs 0.798 in Mode A)
else:
    S8_eu = nb03['sigma8']['S8_eu']

print(f'  Omega_m(LCDM) = {Omega_m_lcdm:.4f} (Planck input)')
print(f'  Omega_m(EU)   = {Omega_m_eu:.4f} (CLASS drained)')
print(f'  f_cdm(z=0)    = {f_cdm_survived:.4f} (CDM survived)')

# Observational S8 (KiDS-1000 + DES-Y3 combined)
S8_obs = 0.776      # DES Y3 (Abbott+ 2022, arXiv:2105.13549)
S8_obs_err = 0.017  # DES Y3 (symmetric error)
# Note: KiDS-Legacy (Wright+2025, arXiv:2503.19441) reports S8 = 0.815 ± 0.016,
# a significant upward shift from KiDS-1000 (0.766). The authors attribute this
# to photo-z recalibration (their §5.2). Given the calibration instability,
# we adopt DES-Y3 as the primary S8 reference: larger area (5000 vs 1350 deg²),
# stable calibration across DES analyses, and consistent with HSC-Y3 (0.776).

print('=' * 60)
print('§2. BRIDGE QUANTITIES')
print('=' * 60)
print(f'  r_d(LCDM) = {rd_lcdm:.2f} Mpc')
print(f'  r_d(EU)   = {rd_eu:.2f} Mpc')
print(f'  Delta r_d = {(rd_eu/rd_lcdm - 1)*100:.3f}%')
print(f'  sigma8(LCDM) = {sigma8_lcdm:.4f}')
print(f'  sigma8(EU)   = {sigma8_eu:.4f}')
print(f'  S8(LCDM) = {S8_lcdm:.4f}')
print(f'  S8(EU)   = {S8_eu:.4f}')
print(f'  S8(DES Y3) = {S8_obs} +/- {S8_obs_err}')
print(f'  S8 tension LCDM: {abs(S8_lcdm - S8_obs)/S8_obs_err:.1f} sigma')
print(f'  S8 tension EU:   {abs(S8_eu - S8_obs)/S8_obs_err:.1f} sigma')


---
# §3. CMB Power Spectrum

Compare EU and ΛCDM angular power spectra ($C_\ell^{TT}$, $C_\ell^{EE}$).
Compute $\chi^2$ against Planck 2018 binned data to quantify CMB consistency.

> **Note**: This comparison uses Planck 2018 binned bandpowers for a quick
> visual and $\chi^2$ diagnostic. The full CMB likelihood analysis
> (plik TTTEEE + lowl TT + lowl EE) is performed in NB04v2 via Cobaya MCMC.


In [ ]:
# ============================================================
# §3. CMB POWER SPECTRA vs PLANCK 2018
# ============================================================

# ── Download Planck 2018 TT bandpowers ──
import urllib.request

# Primary: ESA Planck Legacy Archive
# Fallback: Zenodo mirror (persistent DOI)
planck_urls = [
    ('Zenodo',   'https://zenodo.org/records/16283859/files/COM_PowerSpect_CMB-TT-binned_R3.01.txt?download=1'),
    ('PLA/ESA',  'https://pla.esac.esa.int/pla/aio/product-action?COSMOLOGY.FILE_ID=COM_PowerSpect_CMB-TT-binned_R3.01.txt'),
]
planck_file = '/content/planck_tt_binned.txt'
if not os.path.exists(planck_file):
    planck_file_ok = False
    for src_name, url in planck_urls:
        try:
            urllib.request.urlretrieve(url, planck_file)
            print(f'[OK] Planck 2018 TT bandpowers downloaded from {src_name}')
            planck_file_ok = True
            break
        except Exception as e:
            print(f'[WARN] {src_name} failed: {e}')
    assert planck_file_ok, (
        'FATAL: Planck 2018 TT bandpowers download failed from all sources. '
        'Check internet connection or download manually from '
        'https://zenodo.org/records/16283859')

# ── Load CLASS Cl (lensed) ──
cl_lcdm = np.loadtxt(os.path.join(DATA_DIR, 'lcdm_cl_lensed.dat'))
cl_eu   = np.loadtxt(os.path.join(DATA_DIR, 'eu_cl_lensed.dat'))

# CLASS outputs ℓ(ℓ+1)Cℓ/(2π) in DIMENSIONLESS units.
# Planck bandpowers are Dℓ in µK². Convert:
#   Dℓ[µK²] = (dimensionless) × T_CMB² × 1e12
T_CMB = 2.7255  # K, COBE/FIRAS
to_muK2 = T_CMB**2 * 1e12  # = 7.4283 × 10¹²

ell_l = cl_lcdm[:, 0]; tt_l = cl_lcdm[:, 1] * to_muK2
ell_e = cl_eu[:, 0];   tt_e = cl_eu[:, 1] * to_muK2
print(f'  CLASS Dl conversion: dimensionless × {to_muK2:.4e} → µK²')
print(f'  Dl(ℓ=200, LCDM) = {tt_l[198]:.1f} µK² (expected ~5600)')

# ── Load Planck bandpowers ──
planck_ell, planck_Dl, planck_err_lo, planck_err_hi = None, None, None, None
if planck_file and os.path.exists(planck_file):
    # Planck binned format: ell, Dl, -dDl, +dDl, BestFit
    try:
        pdata = np.loadtxt(planck_file)
        planck_ell = pdata[:, 0]         # ell (bin center)
        planck_Dl = pdata[:, 1]          # D_ell [muK^2]
        planck_err_lo = pdata[:, 2]      # -dDl (lower error)
        planck_err_hi = pdata[:, 3]      # +dDl (upper error)
        print(f'[OK] Planck bandpowers: {len(planck_ell)} bins, ell=[{planck_ell[0]:.0f},{planck_ell[-1]:.0f}]')
    except Exception as e:
        print(f'[WARN] Could not parse Planck file: {e}')

# ── chi2 computation ──
def chi2_vs_planck(ell_theory, Dl_theory, planck_ell, planck_Dl, planck_err):
    """Compute chi2 of theory vs Planck bandpowers."""
    Dl_interp = interp1d(ell_theory, Dl_theory, bounds_error=False,
                         fill_value=np.nan)(planck_ell)
    # Mask out-of-range multipoles
    valid = ~np.isnan(Dl_interp)
    Dl_interp = Dl_interp[valid]
    planck_Dl_v = planck_Dl[valid]
    err_v = planck_err[valid]
    residual = (Dl_interp - planck_Dl_v)
    # Use symmetric error (average of lo/hi)
    err = planck_err
    chi2 = np.sum((residual / err_v)**2)
    return chi2, int(valid.sum())

if planck_ell is not None:
    err_sym = (planck_err_lo + planck_err_hi) / 2
    chi2_lcdm, ndof = chi2_vs_planck(ell_l, tt_l, planck_ell, planck_Dl, err_sym)
    chi2_eu, _      = chi2_vs_planck(ell_e, tt_e, planck_ell, planck_Dl, err_sym)
    delta_chi2 = chi2_eu - chi2_lcdm
    # AIC: delta_AIC = delta_chi2 + 2*delta_k (EU has delta_k = 0)
    delta_AIC = delta_chi2  # zero extra parameters

    print(f'\nchi2(LCDM) = {chi2_lcdm:.1f} / {ndof} dof')
    print(f'chi2(EU)   = {chi2_eu:.1f} / {ndof} dof')
    print(f'Delta chi2 = {delta_chi2:+.1f}')
    print(f'Delta AIC  = {delta_AIC:+.1f} (Delta_k = 0)')
else:
    chi2_lcdm = chi2_eu = delta_chi2 = delta_AIC = None

# ── Figure: CMB TT vs Planck ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8),
                                gridspec_kw={'height_ratios': [3, 1]})

# Top: spectra
ax1.plot(ell_l, tt_l, 'b-', label=r'$\Lambda$CDM', lw=1.5)
ax1.plot(ell_e, tt_e, 'r--', label='EU', lw=1.5)
if planck_ell is not None:
    ax1.errorbar(planck_ell, planck_Dl, yerr=[planck_err_lo, planck_err_hi],
                 fmt='k.', ms=2, elinewidth=0.5, alpha=0.6, label='Planck 2018')
ax1.set_ylabel(r'$D_\ell^{TT}$ [$\mu$K$^2$]')
ax1.set_xlim(2, 2500)
ax1.legend()
ax1.set_title('CMB TT Power Spectrum')

# Bottom: residuals
# EU - LCDM difference
ell_common = np.intersect1d(ell_l.astype(int), ell_e.astype(int))
tt_l_c = interp1d(ell_l, tt_l)(ell_common.astype(float))
tt_e_c = interp1d(ell_e, tt_e)(ell_common.astype(float))
residual = (tt_e_c / tt_l_c - 1) * 100
ax2.plot(ell_common, residual, 'r-', lw=1)
ax2.axhline(0, color='gray', ls=':', lw=0.8)
ax2.set_xlabel(r'$\ell$')
ax2.set_ylabel(r'$(C_\ell^{\rm EU} / C_\ell^{\Lambda{\rm CDM}} - 1)$ [%]')
ax2.set_xlim(2, 2500)
ax2.set_ylim(-5, 5)

plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/fig_NB04_CMB.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/fig_NB04_CMB.pdf', bbox_inches='tight')
plt.show()
print('[OK] CMB figure saved')


---
# §4. ISW Enhancement & CMB Lensing

The EU coupling modifies late-time gravitational potential decay, enhancing the
Integrated Sachs-Wolfe (ISW) effect at low multipoles ($\ell < 30$) while
leaving acoustic peaks unaffected ($\Delta C_\ell < 0.1\%$ for $\ell > 30$).


In [ ]:
# ============================================================
# §4. ISW ENHANCEMENT & CMB LENSING
# ============================================================
# The EU coupling modifies the time evolution of gravitational
# potentials, enhancing the ISW effect at low multipoles.
# ============================================================

# ── ISW: Low-ell power excess ──
# Compare EU vs LCDM at ell < 30 (ISW-dominated)
mask_isw_l = ell_l < 30
mask_isw_e = ell_e < 30

# ISW enhancement ratio
ell_isw = ell_l[mask_isw_l]
tt_l_isw = tt_l[mask_isw_l]
tt_e_isw = interp1d(ell_e, tt_e, bounds_error=True)(ell_isw)
isw_ratio = tt_e_isw / tt_l_isw

print('=' * 60)
print('§4. ISW ENHANCEMENT')
print('=' * 60)
print(f'  Mean ISW ratio (ell < 30): {np.mean(isw_ratio):.4f}')
print(f'  Max ISW enhancement:       {np.max(isw_ratio):.4f}')

# ── Lensing: High-ell smoothing ──
# Load unlensed vs lensed to check lensing consistency
cl_eu_unlensed = np.loadtxt(os.path.join(DATA_DIR, 'eu_cl.dat'))
cl_eu_lensed   = np.loadtxt(os.path.join(DATA_DIR, 'eu_cl_lensed.dat'))

ell_ul = cl_eu_unlensed[:, 0]; tt_ul = cl_eu_unlensed[:, 1] * to_muK2
ell_le = cl_eu_lensed[:, 0];   tt_le = cl_eu_lensed[:, 1] * to_muK2

# Lensing amplitude proxy: A_L ~ ratio of lensed/unlensed smoothing
# At ell ~ 1000-1500, lensing smooths peaks
mask_lens_le = (ell_le > 800) & (ell_le < 1500)
mask_lens_ul = (ell_ul > 800) & (ell_ul < 1500)
peak_ratio_eu = np.std(tt_le[mask_lens_le]) / np.std(tt_ul[mask_lens_ul])

cl_lcdm_unlensed = np.loadtxt(os.path.join(DATA_DIR, 'lcdm_cl.dat'))
ell_ul_l = cl_lcdm_unlensed[:, 0]; tt_ul_l = cl_lcdm_unlensed[:, 1] * to_muK2
cl_lcdm_lensed = np.loadtxt(os.path.join(DATA_DIR, 'lcdm_cl_lensed.dat'))
ell_le_l = cl_lcdm_lensed[:, 0]; tt_le_l = cl_lcdm_lensed[:, 1] * to_muK2
mask_lens_le_l = (ell_le_l > 800) & (ell_le_l < 1500)
mask_lens_ul_l = (ell_ul_l > 800) & (ell_ul_l < 1500)
peak_ratio_lcdm = np.std(tt_le_l[mask_lens_le_l]) / np.std(tt_ul_l[mask_lens_ul_l])

print(f'  Lensing smoothing (LCDM): {peak_ratio_lcdm:.4f}')
print(f'  Lensing smoothing (EU):   {peak_ratio_eu:.4f}')

# ── Figure: ISW + Lensing ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: ISW (low-ell)
ax1.plot(ell_l[mask_isw_l], tt_l[mask_isw_l], 'b-o', label=r'$\Lambda$CDM', ms=3)
ax1.plot(ell_isw, tt_e_isw, 'r--s', label='EU', ms=3)
ax1.set_xlabel(r'$\ell$'); ax1.set_ylabel(r'$D_\ell^{TT}$')
ax1.set_title('ISW Enhancement (low $\\ell$)')
ax1.legend()

# Right: Lensing (high-ell peaks)
mask_hi = (ell_le > 600) & (ell_le < 1800)
ax2.plot(ell_le_l[(ell_le_l > 600) & (ell_le_l < 1800)],
         tt_le_l[(ell_le_l > 600) & (ell_le_l < 1800)], 'b-', label=r'$\Lambda$CDM lensed', lw=1)
ax2.plot(ell_le[mask_hi], tt_le[mask_hi], 'r--', label='EU lensed', lw=1)
ax2.set_xlabel(r'$\ell$'); ax2.set_ylabel(r'$D_\ell^{TT}$')
ax2.set_title('CMB Lensing (acoustic peaks)')
ax2.legend()

plt.tight_layout()
plt.savefig('figures/fig_NB04_ISW.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/fig_NB04_ISW.pdf', bbox_inches='tight')
plt.show()
print('[OK] ISW/Lensing figure saved')


---
# §4.1. Lensing Spectral Fingerprint

The EU coupling modifies the gravitational lensing potential through
CDM density suppression. We compute the $\ell$-dependent residual
$\Delta C_\ell / C_\ell$ across ISW, transition, and acoustic regimes
to identify the unique EU spectral signature.


In [ ]:
# ============================================================
# §4.1. LENSING SPECTRAL FINGERPRINT
# ============================================================

# Spectral bands
bands = {
    'ISW (l=2-10)':         (2, 10),
    'Low-l (l=2-30)':       (2, 30),
    'Transition (l=30-100)': (30, 100),
    'Acoustic (l=100-800)':  (100, 800),
    'Damping (l=800-2500)':  (800, 2500),
}

# Compute fractional residuals per band
print('=' * 60)
print('§4.1. LENSING SPECTRAL FINGERPRINT')
print('=' * 60)
print(f'  {"Band":<25} {"Mean dC/C [%]":>14} {"Max |dC/C| [%]":>16}')
print('  ' + '-' * 56)

# Use lensed spectra
mask_common = (ell_l >= 2) & (ell_l <= 2500)
ell_c = ell_l[mask_common]
tt_lcdm_c = tt_l[mask_common]
tt_eu_c = interp1d(ell_e, tt_e, bounds_error=False, fill_value=0)(ell_c)
residual = (tt_eu_c / tt_lcdm_c - 1) * 100  # percent

for name, (l_min, l_max) in bands.items():
    mask = (ell_c >= l_min) & (ell_c <= l_max)
    if mask.sum() == 0: continue
    mean_r = np.mean(residual[mask])
    max_r = np.max(np.abs(residual[mask]))
    print(f'  {name:<25} {mean_r:>+14.3f} {max_r:>16.3f}')

# Figure
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ell_c, residual, 'r-', lw=0.8, alpha=0.7)
ax.axhline(0, color='k', ls='--', alpha=0.5)
ax.axhspan(-1, 1, alpha=0.08, color='green', label=r'$\pm 1\%$ band')
ax.axvline(30, color='blue', ls=':', alpha=0.4, label='ISW boundary')
ax.set_xlabel(r'$\ell$')
ax.set_ylabel(r'$\Delta C_\ell^{TT} / C_\ell^{TT}$ [%]')
ax.set_title('EU Lensing Spectral Fingerprint')
ax.set_xlim(2, 2500); ax.set_xscale('log')
ax.legend()
plt.tight_layout()
plt.savefig('figures/fig_NB04_fingerprint.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] Fingerprint figure saved')


---
# §4.2. CMB Interpretation: Why EU Preserves the Acoustic Peaks

The EU coupling $\Gamma = \varepsilon(z)\,H$ activates only at
$z < z_{\rm trans} \approx 6$, **well after recombination** ($z_{\rm rec} \approx 1100$).
The CMB anisotropies are set at recombination and propagate freely —
the late-time coupling affects only:

1. **ISW effect** ($\ell \lesssim 30$): Enhanced by $\sim 3\%$ due to faster
   potential decay. This regime is **cosmic-variance dominated** and
   undetectable with current data.

2. **Lensing** ($\ell \gtrsim 100$): Slightly suppressed due to reduced CDM
   clustering. The effect is $< 1\%$ on the TT spectrum, well within
   Planck error bars.

3. **Acoustic peaks** ($\ell \sim 200{-}2000$): **Unaffected** ($< 0.1\%$).
   The peaks are frozen at recombination and the EU coupling does not
   modify the sound horizon $r_d$ (verified: $r_d^{\rm EU} = r_d^{\rm LCDM}$
   to machine precision).

> **Result**: The EU model is **CMB-safe by construction**.
> No fine-tuning is needed to preserve the acoustic peaks.


---
---
# Part III — Structure Formation

Full-shape clustering (DESI DR1 ShapeFit), growth rate, and perturbation stability.
The $S_8$ tension resolution emerges naturally from CDM evaporation.


---
# §5. Full-Shape Clustering — DESI DR1 ShapeFit

AP-resolved validation using DESI DR1 ShapeFit compressed parameters
($q_{\rm iso}$, $q_{\rm AP}$, $df$, $dm$) with full 4×4 covariance.

- **Data**: Official DESI DR1 HDF5 likelihoods (6 redshift bins)
- **Model**: CLASS-EU exact $H(z)$, $D_M(z)$ backgrounds + growth ODE
- **Fiducial**: Planck 2018 ΛCDM (= DESI AbacusSummit), no hardcoded constants
- **Reference**: Brieden+2021 (arXiv:2106.11930), DESI 2024 V

> **Note on DESI DR1 vs DR2**: This section uses DESI DR1 full-shape
> ShapeFit likelihoods (the only publicly available full-shape data as of 2025).
> §7.1 supplements this with DESI DR2 BAO distance measurements
> (from CobayaSampler/bao\_data). The NB04v2 MCMC also uses DR2 BAO.
> When DESI DR2 full-shape likelihoods become public, this section
> should be updated accordingly.


### §5 Methodology: Growth ODE vs raw CLASS P(k)

The following cell computes f×σ₈(z) using the **linear growth ODE** rather than the raw CLASS P(k).
This is a deliberate physical choice, not an approximation:

**Why not use σ₈ directly from CLASS P(k)?**

The raw CLASS P(k) gives σ₈(EU) > σ₈(ΛCDM), which would imply
the EU model *amplifies* growth — physically incorrect for a model that drains CDM mass.
The inflation comes from two well-documented artifacts:

1. **h⁻¹ unit rescaling**: With h_EU > h_ΛCDM (via θ_s shooting), the 8 h⁻¹ Mpc sphere is
   physically smaller in EU, capturing more small-scale power. The early-universe physics
   is identical (same ω_cdm, ω_b from Planck).

2. **Spurious vacuum clustering**: CLASS treats DE as a fluid (Valiviita et al. 2008) that
   develops perturbations δ_fld. In the EU model, DE is the quantum vacuum (Λ with decay),
   which is homogeneous sub-horizon. The fluid clustering adds unphysical gravity via the
   Poisson equation. The CDM perturbation source cancels exactly (Q ∝ ρ_c), but the FLD
   perturbation does not — this is correct for generic IDE but unphysical for vacuum decay.

**The growth ODE solution:**

The sub-horizon quasi-static growth equation D'' + (2+H'/H)D' - 3/2 Ω_m(z) D = 0
captures the correct physics: same initial conditions (A_s, n_s), same transfer function,
but with Ω_m(z) reduced by the CDM drain. The corrected σ₈ and S₈ values are computed
below and compared against DES-Y3 (0.776 ± 0.017).

Reference: Valiviita, Majerotto & Maartens (2008), JCAP 07, 020.

In [ ]:
# ============================================================
# §5. DESI DR1 FULL-SHAPE — ShapeFit (4x4, AP-resolved)
# ============================================================
# Data: DESI DR1 official ShapeFit compressed likelihoods
# Source: https://data.desi.lbl.gov/public/dr1/vac/dr1/
#         full-shape-bao-clustering/v1.0/data/likelihood/
# Method: Brieden+2021 (arXiv:2106.11930), DESI 2024 V
# Covariance: Full 4x4 (q_iso, q_ap, df, dm)
# Background: CLASS-EU and CLASS-LCDM exact tables
# ============================================================

import h5py
from scipy.integrate import quad as scipy_quad, solve_ivp
from scipy.interpolate import CubicSpline

c_kms = 299792.458

# ── 1. Fiducial cosmology ──
# ALL values from upstream cells — NO FALLBACKS, NO HARDCODED CONSTANTS
# H0, Omega_m: from NB01 params (Planck 2018 = DESI AbacusSummit fiducial)
# rd: from §2 bridge cell (extracted from CLASS background)
# sigma8: from §2 bridge cell (computed from CLASS P(k))
# Omega_m_eu: from §2 bridge cell (extracted from CLASS EU background)
# Fiducial params: Mode B uses MCMC C2, Mode A uses Planck LCDM
if USE_MCMC_PARAMS:
    H0_fid = C2_DERIVED['H0']         # 68.886 km/s/Mpc
    Om_fid = C2_DERIVED['Omega_m']    # 0.2883
else:
    H0_fid = params['planck2018_LCDM_derived']['H0']         # km/s/Mpc
    Om_fid = params['planck2018_LCDM_derived']['Omega_m']    # LCDM = DESI fiducial

assert rd_lcdm is not None, 'FATAL: rd_lcdm not set by §2 bridge. Re-run NB02.'
assert rd_eu is not None,   'FATAL: rd_eu not set by §2 bridge. Re-run NB02.'
rd_fid = rd_lcdm  # LCDM rd = DESI fiducial rd
rd_EU  = rd_eu    # EU rd (from CLASS EU)

if USE_MCMC_PARAMS:
    sigma8_fid = C2_DERIVED['sigma8']  # 0.8274
    print(f'  [MODE B] ShapeFit fiducial: H0={H0_fid:.2f}, Om={Om_fid:.4f}, σ8={sigma8_fid:.4f}')
else:
    sigma8_fid = sigma8_lcdm  # from §2 bridge

print(f'  Fiducial (from §2 bridge + NB01):')
print(f'    H0      = {H0_fid} km/s/Mpc')
print(f'    Omega_m = {Om_fid}')
print(f'    rd_fid  = {rd_fid:.6f} Mpc')
print(f'    rd_EU   = {rd_EU:.6f} Mpc')
print(f'    sigma8  = {sigma8_fid:.4f}')
print(f'    Omega_m_eu = {Omega_m_eu:.6f} (from CLASS EU)')

print(f'  Fiducial (from CLASS LCDM, = DESI AbacusSummit):')
print(f'    H0     = {H0_fid:.2f} km/s/Mpc')
print(f'    Omega_m = {Om_fid:.4f}')  
print(f'    rd     = {rd_fid:.4f} Mpc')
print(f'    sigma8 = {sigma8_fid:.4f}')

# ── 2. Load CLASS backgrounds ──
def load_class_bg_columns(filename):
    data = np.loadtxt(os.path.join(DATA_DIR, filename))
    z = data[:, 0]; H_inv = data[:, 3]; DM = data[:, 4]
    idx = np.argsort(z)
    return z[idx], H_inv[idx] * c_kms, DM[idx]

z_bg_eu,   H_bg_eu,   DM_bg_eu   = load_class_bg_columns('eu_background.dat')
z_bg_lcdm, H_bg_lcdm, DM_bg_lcdm = load_class_bg_columns('lcdm_background.dat')

# ── 3. Fiducial H(z) and D_M(z) from CLASS LCDM (exact, no analytic) ──
H_fid_interp  = CubicSpline(z_bg_lcdm, H_bg_lcdm)
DM_fid_interp = CubicSpline(z_bg_lcdm, DM_bg_lcdm)

def H_fid(z):  return float(H_fid_interp(z))
def DM_fid(z): return float(DM_fid_interp(z))

# ── 4. ShapeFit q_iso, q_ap from CLASS ──
def compute_shapefit_q(z_eff, z_arr, H_arr, DM_arr, rd_model):
    H_m  = np.interp(z_eff, z_arr, H_arr)
    DM_m = np.interp(z_eff, z_arr, DM_arr)
    DH_m = c_kms / H_m
    DH_f = c_kms / H_fid(z_eff)
    DM_f = DM_fid(z_eff)
    q_par  = (DH_m / rd_model) / (DH_f / rd_fid)
    q_perp = (DM_m / rd_model) / (DM_f / rd_fid)
    q_iso = (q_par * q_perp**2)**(1.0/3.0)
    q_ap  = q_par / q_perp
    return q_iso, q_ap

# ── 5. df prediction (growth rate) ──
# Solve growth ODE for LCDM and EU → compute f*sigma8 ratio
z_gr = np.linspace(0.0, 2.5, 300)

def growth_ode(model, Om_interp_func=None, H_interp=None, Hp_interp=None):
    def sys(lna, y):
        a = np.exp(lna); z = 1/a - 1; delta, dp = y
        if model == 'LCDM' or Om_interp_func is None:
            Om_a = Om_fid * a**(-3) / (Om_fid * a**(-3) + (1-Om_fid))
            HpH = -1.5 * Om_a
        else:
            if 0 <= z <= 100:
                Om_a = float(Om_interp_func(z))
                HpH  = -(1+z) * float(Hp_interp(z)) / float(H_interp(z))
            else:
                Om_a = Om_fid * a**(-3) / (Om_fid * a**(-3) + (1-Om_fid))
                HpH = -1.5 * Om_a
        return [dp, -(2.0 + HpH)*dp + 1.5*Om_a*delta]
    a0 = 1e-3
    sol = solve_ivp(sys, [np.log(a0), 0], [a0, a0],
                    max_step=0.01, rtol=1e-10, atol=1e-12, dense_output=True)
    return sol

sol_lcdm_g = growth_ode('LCDM')

# EU: build splines for Om(z), H(z), H'(z) from CLASS
z_mask = z_e <= 110
z_eu_s = z_e[z_mask]; H_eu_s = H_e[z_mask] * c_kms
si = np.argsort(z_eu_s); z_eu_s = z_eu_s[si]; H_eu_s = H_eu_s[si]
um = np.concatenate(([True], np.diff(z_eu_s) > 0))
z_eu_s = z_eu_s[um]; H_eu_s = H_eu_s[um]
H_eu_spl = CubicSpline(z_eu_s, H_eu_s)
Hp_eu_spl = H_eu_spl.derivative()

rho_cdm_e = bg_eu[:, cols_e['(.)rho_cdm']]
rho_b_e   = bg_eu[:, cols_e['(.)rho_b']]
rho_cr_e  = bg_eu[:, cols_e['(.)rho_crit']]
# Include neutrinos in Omega_m(z) — NB03 Fix #2b
assert '(.)rho_ncdm[0]' in cols_e, "FATAL: (.)rho_ncdm[0] not in CLASS EU background columns"
rho_ncdm_e = bg_eu[:, cols_e['(.)rho_ncdm[0]']]
Om_ez = ((rho_cdm_e + rho_b_e + rho_ncdm_e) / rho_cr_e)[z_mask][si][um]
Om_eu_spl = CubicSpline(z_eu_s, Om_ez)

sol_eu_g = growth_ode('EU', Om_eu_spl, H_eu_spl, Hp_eu_spl)

# Extract f(z), D(z)
D_lcdm_arr = np.array([sol_lcdm_g.sol(np.log(1/(1+z)))[0] for z in z_gr])
D_eu_arr   = np.array([sol_eu_g.sol(np.log(1/(1+z)))[0] for z in z_gr])
growth_suppression = D_eu_arr[0] / D_lcdm_arr[0]
D_lcdm_arr /= D_lcdm_arr[0]; D_eu_arr /= D_eu_arr[0]
f_lcdm_arr = np.array([sol_lcdm_g.sol(np.log(1/(1+z)))[1]/sol_lcdm_g.sol(np.log(1/(1+z)))[0] for z in z_gr])
f_eu_arr   = np.array([sol_eu_g.sol(np.log(1/(1+z)))[1]/sol_eu_g.sol(np.log(1/(1+z)))[0] for z in z_gr])

sigma8_eu_corr = sigma8_fid * growth_suppression
fsig8_lcdm_arr = f_lcdm_arr * sigma8_fid * D_lcdm_arr
fsig8_eu_arr   = f_eu_arr * sigma8_eu_corr * D_eu_arr

print(f'  Growth suppression = {growth_suppression:.4f}')
print(f'  sigma8_EU corrected = {sigma8_eu_corr:.4f}')

def df_prediction(z_eff, model='EU'):
    """df = f*sigma8(model) / f*sigma8(fiducial)"""
    fs8_fid = np.interp(z_eff, z_gr, fsig8_lcdm_arr)
    if model == 'EU':
        fs8_mod = np.interp(z_eff, z_gr, fsig8_eu_arr)
    else:
        fs8_mod = fs8_fid
    return fs8_mod / fs8_fid

# ── 6. dm prediction (shape parameter) ──
pk_lcdm_dat = np.loadtxt(os.path.join(DATA_DIR, 'lcdm_pk.dat'))
pk_eu_dat   = np.loadtxt(os.path.join(DATA_DIR, 'eu_pk.dat'))
cs_pk_eu = CubicSpline(np.log(pk_eu_dat[:,0]), np.log(pk_eu_dat[:,1]))
cs_pk_lc = CubicSpline(np.log(pk_lcdm_dat[:,0]), np.log(pk_lcdm_dat[:,1]))
k_piv = 0.03  # h/Mpc pivot (ShapeFit convention)
dm_eu_global = cs_pk_eu(np.log(k_piv), 1) - cs_pk_lc(np.log(k_piv), 1)
print(f'  dm_EU (P(k) tilt at k={k_piv}) = {dm_eu_global:.4f}')

# ── 7. DESI bins and files ──
desi_bins = [
    ('BGS',  0.295, 'likelihood_shapefit_spectrum-poles-rotated+bao-recon_syst-rotation-hod-photo_BGS_BRIGHT-21.5_GCcomb_z0.1-0.4_thetacut0.05.h5'),
    ('LRG1', 0.510, 'likelihood_shapefit_spectrum-poles-rotated+bao-recon_syst-rotation-hod-photo_LRG_GCcomb_z0.4-0.6_thetacut0.05.h5'),
    ('LRG2', 0.706, 'likelihood_shapefit_spectrum-poles-rotated+bao-recon_syst-rotation-hod-photo_LRG_GCcomb_z0.6-0.8_thetacut0.05.h5'),
    ('LRG3', 0.930, 'likelihood_shapefit_spectrum-poles-rotated+bao-recon_syst-rotation-hod-photo_LRG_GCcomb_z0.8-1.1_thetacut0.05.h5'),
    ('ELG',  1.317, 'likelihood_shapefit_spectrum-poles-rotated+bao-recon_syst-rotation-hod-photo_ELG_LOPnotqso_GCcomb_z1.1-1.6_thetacut0.05.h5'),
    ('QSO',  1.491, 'likelihood_shapefit_spectrum-poles-rotated+bao-recon_syst-rotation-hod-photo_QSO_GCcomb_z0.8-2.1_thetacut0.05.h5'),
]

DESI_DATA_DIR = 'dr1_full_tests/data'
os.makedirs(DESI_DATA_DIR, exist_ok=True)
DESI_URL = 'https://data.desi.lbl.gov/public/dr1/vac/dr1/full-shape-bao-clustering/v1.0/data/likelihood'
for name, z_eff, fn in desi_bins:
    fp = os.path.join(DESI_DATA_DIR, fn)
    if not os.path.exists(fp):
        print(f'  Downloading {name}...', end=' ', flush=True)
        import urllib.request
        urllib.request.urlretrieve(f'{DESI_URL}/{fn}', fp)
        print('OK')

# ── 8. Compute theory vectors and chi2 (FULL 4x4) ──
chi2_eu_tot = 0.0; chi2_lc_tot = 0.0; n_dof = 0
results = []

print(f'\n  {"Bin":6s} {"z":>5s}  {"chi2_EU":>8s}  {"chi2_LC":>8s}  {"Dchi2":>8s}')
print(f'  {"-"*6} {"-"*5}  {"-"*8}  {"-"*8}  {"-"*8}')

for name, z_eff, fn in desi_bins:
    with h5py.File(os.path.join(DESI_DATA_DIR, fn), 'r') as hf:
        obs = hf['observable']['shapefit']
        pnames = [x.decode() for x in obs['labels_values'][()]]
        data = np.array([obs[p]['value'][0] for p in pnames])
        cov = hf['covariance']['value'][()]
    
    inv_cov = np.linalg.inv(cov)
    
    # EU theory vector: [q_iso, q_ap, df, dm]
    qi_eu, qa_eu = compute_shapefit_q(z_eff, z_bg_eu, H_bg_eu, DM_bg_eu, rd_EU)
    df_eu = df_prediction(z_eff, 'EU')
    dm_eu = dm_eu_global  # z-independent at linear level
    th_eu = np.array([qi_eu, qa_eu, df_eu, dm_eu])
    
    # LCDM theory: q=1, df=1, dm=0 by construction (model = fiducial)
    qi_lc, qa_lc = compute_shapefit_q(z_eff, z_bg_lcdm, H_bg_lcdm, DM_bg_lcdm, rd_fid)
    th_lc = np.array([qi_lc, qa_lc, 1.0, 0.0])
    
    d_eu = data - th_eu
    d_lc = data - th_lc
    chi2_sf_bin_eu = float(d_eu @ inv_cov @ d_eu)
    chi2_sf_bin_lc = float(d_lc @ inv_cov @ d_lc)
    
    chi2_eu_tot += chi2_sf_bin_eu; chi2_lc_tot += chi2_sf_bin_lc; n_dof += 4
    
    results.append(dict(name=name, z_eff=z_eff, pnames=pnames,
                        data=data, th_eu=th_eu, th_lc=th_lc,
                        cov_diag=np.sqrt(np.diag(cov)),
                        chi2_eu=chi2_sf_bin_eu, chi2_lc=chi2_sf_bin_lc))
    
    print(f'  {name:6s} {z_eff:5.3f}  {chi2_sf_bin_eu:8.3f}  {chi2_sf_bin_lc:8.3f}  {chi2_sf_bin_eu-chi2_sf_bin_lc:+8.3f}')

print(f'\n  TOTAL ({len(desi_bins)} bins, {n_dof} data points):')
print(f'    chi2_EU   = {chi2_eu_tot:.3f}  (chi2/dof = {chi2_eu_tot/n_dof:.3f})')
print(f'    chi2_LCDM = {chi2_lc_tot:.3f}  (chi2/dof = {chi2_lc_tot/n_dof:.3f})')
print(f'    Delta_chi2 = {chi2_eu_tot - chi2_lc_tot:+.3f}')
fav = 'EU' if chi2_eu_tot < chi2_lc_tot else 'LCDM'
print(f'    ==> {fav} FAVORED by {abs(chi2_eu_tot - chi2_lc_tot):.2f}')

# ── 9. Store for §6 and §12 ──
desi_results = dict(
    chi2_EU=chi2_eu_tot, chi2_LCDM=chi2_lc_tot,
    delta_chi2=chi2_eu_tot - chi2_lc_tot,
    n_bins=len(desi_bins), n_dof=n_dof,
    growth_suppression=growth_suppression,
    sigma8_eu=sigma8_eu_corr, sigma8_lcdm=sigma8_fid,
    bins=results
)

# Update sigma8_eu for downstream cells
sigma8_eu = sigma8_eu_corr
# Omega_m_eu already defined in §2 bridge (from CLASS EU background, exact)
S8_eu = sigma8_eu * np.sqrt(Omega_m_eu / 0.3)
S8_tension_eu = abs(S8_eu - S8_obs) / S8_obs_err
print(f'\n  sigma8(EU) = {sigma8_eu:.4f}, S8(EU) = {S8_eu:.4f}')
print(f'  S8 tension EU: {S8_tension_eu:.2f} sigma (DES Y3)')

# ── 10. Figures ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
param_labels = [r'$q_{\rm iso}$', r'$q_{\rm AP}$', r'$df$', r'$dm$']
for pi in range(4):
    ax = axes[pi//2, pi%2]
    zz = [r['z_eff'] for r in results]
    dd = [r['data'][pi] for r in results]
    ee = [r['cov_diag'][pi] for r in results]
    te = [r['th_eu'][pi] for r in results]
    tl = [r['th_lc'][pi] for r in results]
    ax.errorbar(zz, dd, ee, fmt='ko', ms=6, capsize=4, label='DESI DR1', zorder=3)
    ax.plot(zz, te, 'rs--', ms=8, label='EU', zorder=2)
    ax.plot(zz, tl, 'b^:', ms=8, label=r'$\Lambda$CDM', zorder=2)
    ref = 1.0 if pi < 3 else 0.0
    ax.axhline(ref, color='gray', ls=':', lw=0.8)
    ax.set_xlabel('$z_{\\rm eff}$')
    ax.set_ylabel(param_labels[pi])
    ax.legend(fontsize=9)

fig.suptitle(f'DESI DR1 ShapeFit (4x4): $\\chi^2_{{\\rm EU}}$={chi2_eu_tot:.1f}  '
             f'$\\chi^2_{{\\Lambda\\rm CDM}}$={chi2_lc_tot:.1f}  '
             f'$\\Delta\\chi^2$={chi2_eu_tot-chi2_lc_tot:+.1f}', fontsize=14)
plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/fig_NB04_DESI_ShapeFit.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/fig_NB04_DESI_ShapeFit.pdf', bbox_inches='tight')
plt.show()
print('[OK] DESI ShapeFit 4x4 figure saved')

# Keep P(k) theory plot
pk_l = np.loadtxt(os.path.join(DATA_DIR, 'lcdm_pk.dat'))
pk_e = np.loadtxt(os.path.join(DATA_DIR, 'eu_pk.dat'))
fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={'height_ratios': [3,1]})
kp = np.logspace(np.log10(max(pk_l[0,0],pk_e[0,0])), np.log10(min(pk_l[-1,0],pk_e[-1,0])), 300)
ax1.loglog(kp, np.interp(kp, pk_l[:,0], pk_l[:,1]), 'b-', lw=1.5, label=r'$\Lambda$CDM')
ax1.loglog(kp, np.interp(kp, pk_e[:,0], pk_e[:,1]), 'r--', lw=1.5, label='EU')
ax1.set_ylabel('P(k) [Mpc$^3$]'); ax1.legend(); ax1.set_title('CLASS: Linear P(k)')
ratio = np.interp(kp, pk_e[:,0], pk_e[:,1]) / np.interp(kp, pk_l[:,0], pk_l[:,1])
ax2.semilogx(kp, (ratio-1)*100, 'r-', lw=1.5)
ax2.axhline(0, color='gray', ls=':', lw=0.8)
ax2.set_xlabel('k [1/Mpc]'); ax2.set_ylabel('EU/LCDM - 1 [%]')
plt.tight_layout()
plt.savefig('figures/fig_NB04_Pk_theory.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# COMPATIBILITY: export variables needed by downstream cells
# (§7 BAO, §8 SNe, §9 CC, §10 NoGo, §11 H0, §12 Model Sel.)
# ============================================================
# H(z) functions — from CLASS splines already built above
H_lcdm_func = lambda z: float(H_fid_interp(z))   # H_LCDM(z) [km/s/Mpc]
H_eu_interp = lambda z: float(np.interp(z, z_bg_eu, H_bg_eu))  # H_EU(z)

# Sound horizons — for BAO cells
rd_lcdm = rd_fid
rd_eu   = rd_EU

# Growth quantities — for §10 NoGo, §11 H0 decomposition
# f_lcdm, f_eu, D_lcdm, D_eu already defined as arrays over z_gr
# Make interpolators for downstream cells that need f(z)
f_lcdm_interp = interp1d(z_gr, f_lcdm_arr, bounds_error=False, fill_value='extrapolate')
f_eu_interp   = interp1d(z_gr, f_eu_arr, bounds_error=False, fill_value='extrapolate')
f_lcdm = f_lcdm_arr  # alias
f_eu   = f_eu_arr     # alias
D_lcdm = D_lcdm_arr   # alias  
D_eu   = D_eu_arr     # alias

print(f'  [OK] Exported: H_lcdm_func, H_eu_interp, rd_lcdm, rd_eu, f_lcdm, f_eu')


---
# §6. Growth Rate $f\sigma_8(z)$ — Diagnostic

Visual diagnostic of $f\sigma_8(z)$ curves from §5 growth ODE.
The $\chi^2$ for growth is already included in §5 via the `df` parameter
of the full 4×4 ShapeFit covariance. This cell is for visualization only.


In [ ]:
# ============================================================
# §6. GROWTH RATE f*sigma8(z) — DIAGNOSTIC PLOT
# ============================================================
# No separate chi2: growth is already in §5 via ShapeFit df.
# This cell visualizes the f*sigma8(z) curves and DESI data.
# ============================================================

# Extract f*sigma8 data from §5 results
z_desi_fs = [r['z_eff'] for r in results]
fsig8_data = [r['data'][2] * np.interp(r['z_eff'], z_gr, fsig8_lcdm_arr) for r in results]
fsig8_err  = [r['cov_diag'][2] * np.interp(r['z_eff'], z_gr, fsig8_lcdm_arr) for r in results]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(z_gr, fsig8_lcdm_arr, 'b-', label=r'$\Lambda$CDM', lw=2)
ax.plot(z_gr, fsig8_eu_arr, 'r--', label='EU', lw=2)
ax.errorbar(z_desi_fs, fsig8_data, fsig8_err, fmt='D', color='#E63946', ms=7,
            capsize=4, label='DESI DR1 ShapeFit', alpha=0.9, zorder=3)
ax.set_xlabel('$z$', fontsize=13)
ax.set_ylabel(r'$f\sigma_8(z)$', fontsize=13)
ax.set_title(r'Growth rate $f\sigma_8(z)$ — DESI DR1 (diagnostic)', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/fig_NB04_fsigma8_DESI.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/fig_NB04_fsigma8_DESI.pdf', bbox_inches='tight')
plt.show()
print('[OK] Growth rate diagnostic saved')
print(f'  Note: chi2 for growth already included in §5 (4x4 ShapeFit)')

---
# §6.1. Perturbation Robustness (Valiviita+2008)

For the EU coupling $Q^\mu = -\varepsilon H \rho_{\rm cdm} u^\mu_{\rm cdm}$,
the linear perturbation equations receive a drain term
$\dot{\delta}_{\rm cdm} \supset -\varepsilon H \delta_{\rm cdm}$
(Valiviita, Majerotto & Maartens 2008).

We verify that this correction is **subdominant** ($< 0.5\%$ on $\sigma_8$)
and that the $S_8$ resolution is driven by background kinematics (CDM evaporation),
not IDE perturbation effects. The doom factor $d \equiv 1$ (NB02 §3.3.1)
ensures perturbative stability.


---
---
# Part IV — Distance Probes

Independent validation against BAO, Type Ia Supernovae, and
Cosmic Chronometers. The EU model must match distance data
without fine-tuning ($\Delta k = 0$).


---
# §7. BAO Distance Predictions

Compare EU and $\Lambda$CDM predictions for BAO observables:
$D_V(z)/r_d$, $D_M(z)/r_d$, $D_H(z)/r_d$.


In [ ]:
# ============================================================
# §7. BAO DISTANCE PREDICTIONS
# ============================================================

c_kms = 299792.458  # km/s

# Sound horizon (from §2)
print('=' * 60)
print('§7. BAO DISTANCES')
print('=' * 60)
if rd_lcdm is not None:
    print(f'  r_d(LCDM) = {rd_lcdm:.2f} Mpc')
    print(f'  r_d(EU)   = {rd_eu:.2f} Mpc')
else:
    print('  [WARN] r_d not available from CLASS — using Planck 2018 value')
    assert False, 'FATAL: r_d not in CLASS output. Re-run NB02 with write background = yes'

# BAO observables at DESI redshifts
z_bao = np.array([0.295, 0.510, 0.706, 0.919, 1.317, 1.491])
bao_labels = ['BGS', 'LRG1', 'LRG2', 'LRG3', 'ELG2', 'QSO']

def bao_observables(z_arr, H_func, r_d):
    results = []
    for z in z_arr:
        D_H = c_kms / H_func(z)  # Mpc
        D_M, _ = quad(lambda zp: c_kms / H_func(zp), 0, z)  # Mpc
        D_V = (z * D_H * D_M**2)**(1./3.)
        results.append({'DV_rd': D_V/r_d, 'DM_rd': D_M/r_d, 'DH_rd': D_H/r_d})
    return results

bao_lcdm = bao_observables(z_bao, H_lcdm_func, rd_lcdm)
bao_eu = bao_observables(z_bao, H_eu_interp, rd_eu)

print(f'\n{"Tracer":>6} {"z":>5} | {"DV/rd LCDM":>10} {"DV/rd EU":>10} {"diff%":>6}')
print('-' * 50)
for i, z in enumerate(z_bao):
    diff = (bao_eu[i]['DV_rd'] / bao_lcdm[i]['DV_rd'] - 1) * 100
    print(f'{bao_labels[i]:>6} {z:5.3f} | {bao_lcdm[i]["DV_rd"]:10.3f} {bao_eu[i]["DV_rd"]:10.3f} {diff:+5.2f}%')


### §7.1. BAO $\chi^2$ — DESI DR2 validation

Compare EU and $\Lambda$CDM predictions directly against DESI DR2 data
(consolidated `ALL_GCcomb` file from CobayaSampler/bao_data).

In [ ]:
# ============================================================
# §7.1. BAO chi2 vs DESI DR2 data
# Source: DESI DR2 (2025), arXiv:2503.14738
# ============================================================

# Download DESI DR2 consolidated data vector
import urllib.request
DESI_BAO_URL = 'https://raw.githubusercontent.com/CobayaSampler/bao_data/master/desi_bao_dr2/'
MEAN_FILE = 'desi_gaussian_bao_ALL_GCcomb_mean.txt'
COV_FILE  = 'desi_gaussian_bao_ALL_GCcomb_cov.txt'
_mean_path = f'data_real/{MEAN_FILE}'
_cov_path  = f'data_real/{COV_FILE}'
os.makedirs('data_real', exist_ok=True)

if not os.path.exists(_mean_path):
    urllib.request.urlretrieve(DESI_BAO_URL + MEAN_FILE, _mean_path)
if not os.path.exists(_cov_path):
    urllib.request.urlretrieve(DESI_BAO_URL + COV_FILE, _cov_path)
assert os.path.exists(_mean_path), 'FATAL: DESI DR2 mean download failed'

# Parse data vector
desi_z, desi_val, desi_qty = [], [], []
with open(_mean_path) as f:
    for line in f:
        if line.startswith('#') or not line.strip(): continue
        parts = line.split()
        desi_z.append(float(parts[0]))
        desi_val.append(float(parts[1]))
        desi_qty.append(parts[2])
desi_z = np.array(desi_z)
desi_val = np.array(desi_val)
N_desi = len(desi_z)

# Parse covariance
desi_cov = np.loadtxt(_cov_path).reshape(N_desi, N_desi)
desi_cov_inv = np.linalg.inv(desi_cov)

def chi2_bao_desi(H_func, r_d):
    th = np.zeros(N_desi)
    z_unique = sorted(set(desi_z))
    dm_cache = {}
    for z in z_unique:
        dm_cache[z], _ = quad(lambda zp: c_kms / H_func(zp), 0, z)
    for k in range(N_desi):
        z = desi_z[k]
        if desi_qty[k] == 'DM_over_rs':
            th[k] = dm_cache[z] / r_d
        elif desi_qty[k] == 'DH_over_rs':
            th[k] = c_kms / (H_func(z) * r_d)
        elif desi_qty[k] == 'DV_over_rs':
            DH = c_kms / H_func(z)
            DV = (z * DH * dm_cache[z]**2)**(1.0/3.0)
            th[k] = DV / r_d
        else:
            raise ValueError(f'Unknown: {desi_qty[k]}')
    delta = desi_val - th
    return float(delta @ desi_cov_inv @ delta)

chi2_bao_lcdm = chi2_bao_desi(H_lcdm_func, rd_lcdm)
chi2_bao_eu = chi2_bao_desi(H_eu_interp, rd_eu)

print(f'BAO chi2 (DESI DR2, {N_desi} data points, {len(set(desi_z))} tracers):')
print(f'  chi2_LCDM = {chi2_bao_lcdm:.2f} (chi2/dof = {chi2_bao_lcdm/N_desi:.2f})')
print(f'  chi2_EU   = {chi2_bao_eu:.2f} (chi2/dof = {chi2_bao_eu/N_desi:.2f})')
print(f'  Delta_chi2 (EU-LCDM) = {chi2_bao_eu - chi2_bao_lcdm:+.2f}')


---
# §8. Type Ia Supernovae (Pantheon+)

Compare EU distance modulus predictions against Pantheon+ (Brout+2022).


In [ ]:
# ============================================================
# §8. PANTHEON+ SNe Ia (Brout+2022, arXiv:2202.04077)
# ============================================================
# Full Pantheon+SH0ES analysis with STAT+SYS covariance matrix.
# Uses the complete 1701-entry catalog + 1701×1701 covariance.
# ============================================================

import urllib.request

c_kms = 299792.458

def distance_modulus(z_arr, H_func):
    mu = np.zeros_like(z_arr, dtype=float)
    for i, z in enumerate(z_arr):
        if z < 1e-5: continue
        d_M, _ = quad(lambda zp: c_kms / H_func(zp), 0, z)
        d_L = (1 + z) * d_M
        mu[i] = 5 * np.log10(d_L) + 25
    return mu

# ── Download data + covariance ──
PANTH_BASE = 'https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR'
panth_dat_path = 'results/pantheon_plus.dat'
panth_cov_path = 'results/pantheon_plus_stat_sys.cov'
os.makedirs('results', exist_ok=True)

for url_suffix, local_path in [
    ('/Pantheon%2BSH0ES.dat', panth_dat_path),
    ('/Pantheon%2BSH0ES_STAT%2BSYS.cov', panth_cov_path),
]:
    if not os.path.exists(local_path):
        urllib.request.urlretrieve(PANTH_BASE + url_suffix, local_path)
        print(f'[OK] Downloaded {os.path.basename(local_path)}')

assert os.path.exists(panth_dat_path), 'FATAL: Pantheon+ data not available'
assert os.path.exists(panth_cov_path), 'FATAL: Pantheon+ covariance not available'

# ── Parse catalog ──
import pandas as pd
panth = pd.read_csv(panth_dat_path, sep=r'\s+', comment='#')
N_total = len(panth)

# ── Parse full covariance matrix ──
with open(panth_cov_path) as f:
    N_cov = int(f.readline().strip())
    cov_flat = np.array([float(x) for x in f.read().split()])
assert N_cov == N_total, f'Covariance size {N_cov} != catalog size {N_total}'
C_full = cov_flat.reshape(N_cov, N_cov)
print(f'  Loaded: {N_total} SNe, covariance {N_cov}x{N_cov} (STAT+SYS)')

# ── Apply redshift cut (z > 0.01) — standard Pantheon+ cosmology cut ──
mask = panth['zHD'].values > 0.01
z_panth = panth['zHD'].values[mask]
mu_obs = panth['MU_SH0ES'].values[mask]
is_cal = panth['IS_CALIBRATOR'].values[mask]

# Slice covariance to match cut
idx_keep = np.where(mask)[0]
C_cut = C_full[np.ix_(idx_keep, idx_keep)]
C_inv = np.linalg.inv(C_cut)
N_sn = len(z_panth)
print(f'  After z > 0.01 cut: {N_sn} SNe')
print(f'  Calibrators in sample: {int(np.sum(is_cal))}')

# ── Theory predictions ──
mu_eu_data = distance_modulus(z_panth, H_eu_interp)
mu_lcdm_data = distance_modulus(z_panth, H_lcdm_func)

# ── Analytic marginalization over M_B (Eq. C2, Brout+2022) ──
def chi2_with_M_marginalized(mu_obs, mu_th, C_inv):
    """chi2 with nuisance M analytically marginalized."""
    delta = mu_obs - mu_th
    ones = np.ones(len(delta))
    # M_best = (1^T C^-1 delta) / (1^T C^-1 1)
    A = ones @ C_inv @ delta
    B = ones @ C_inv @ ones
    M_best = A / B
    res = delta - M_best
    chi2 = float(res @ C_inv @ res)
    return chi2, M_best, res

chi2_sn_eu, M_eu, res_eu = chi2_with_M_marginalized(mu_obs, mu_eu_data, C_inv)
chi2_sn_lcdm, M_lcdm, res_lcdm = chi2_with_M_marginalized(mu_obs, mu_lcdm_data, C_inv)

# mu_err_bin for §9.1 PPD diagnostic (diagonal of covariance)
mu_err_bin = np.sqrt(np.diag(C_cut))

print('=' * 60)
print('§8. DISTANCE MODULUS (Pantheon+)')
print('=' * 60)
print(f'  Data: {N_sn} SNe (Pantheon+SH0ES, Brout+2022)')
print(f'  Covariance: full STAT+SYS ({N_sn}x{N_sn})')
print(f'  chi2 LCDM: {chi2_sn_lcdm:.1f} (chi2/N = {chi2_sn_lcdm/N_sn:.2f})')
print(f'  chi2 EU:   {chi2_sn_eu:.1f} (chi2/N = {chi2_sn_eu/N_sn:.2f})')
print(f'  Delta chi2 (EU-LCDM): {chi2_sn_eu - chi2_sn_lcdm:+.1f}')

# ── Theory comparison curves ──
z_sn = np.linspace(0.01, 2.0, 200)
mu_lcdm = distance_modulus(z_sn, H_lcdm_func)
mu_eu = distance_modulus(z_sn, H_eu_interp)
delta_mu = mu_eu - mu_lcdm
print(f'  max |mu(EU) - mu(LCDM)| = {np.max(np.abs(delta_mu)):.4f} mag')

# ── Figures ──
fig, axes = plt.subplots(3, 1, figsize=(10, 10),
                          gridspec_kw={'height_ratios': [2, 1, 1]})

# Panel 1: Hubble diagram
ax1 = axes[0]
ax1.errorbar(z_panth[::5], mu_obs[::5], mu_err_bin[::5], fmt='.', color='gray',
             alpha=0.3, ms=2, capsize=0, label=f'Pantheon+ ({N_sn} SNe)')
ax1.plot(z_sn, mu_lcdm + M_lcdm, 'b-', label=r'$\Lambda$CDM', lw=1.5)
ax1.plot(z_sn, mu_eu + M_eu, 'r--', label='EU', lw=1.5)
ax1.set_ylabel(r'$\mu(z)$ [mag]'); ax1.set_title('Pantheon+ Hubble Diagram (full STAT+SYS cov)')
ax1.legend(fontsize=10)

# Panel 2: EU-LCDM difference
ax2 = axes[1]
ax2.plot(z_sn, delta_mu * 1000, 'r-', lw=1.5)
ax2.axhline(0, color='gray', ls=':', lw=0.8)
ax2.set_ylabel(r'$\Delta\mu$ [mmag]')

# Panel 3: Standardized residuals
std_res_eu = res_eu / mu_err_bin
ax3 = axes[2]
ax3.scatter(z_panth, std_res_eu, s=1, c='red', alpha=0.3, label='EU residuals')
ax3.axhline(0, color='gray', ls='--')
ax3.set_xlabel('z'); ax3.set_ylabel(r'$({\mu_{\rm obs} - \mu_{\rm EU} - M})/\sigma$')
ax3.set_ylim(-5, 5)
ax3.legend(fontsize=9)

plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/fig_NB04_SN.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] Pantheon+ figure saved')
print(f'  Exported: res_eu, res_lcdm, mu_err_bin → §9.1 PPD')


---
# §9. Cosmic Chronometers $H(z)$

Independent $H(z)$ measurements from differential age dating of
passively-evolving galaxies. 32-point compilation from Moresco+ (2022),
Living Rev. Rel. **25**, 6 ([arXiv:2201.07241](https://arxiv.org/abs/2201.07241)).


In [ ]:
# ============================================================
# §9. COSMIC CHRONOMETERS H(z) — Moresco+ (2022)
# ============================================================

# 32-point compilation (Table 3, arXiv:2201.07241)
HZ_Z = np.array([0.07, 0.09, 0.12, 0.17, 0.1791, 0.1993, 0.20, 0.27, 0.28,
                  0.3519, 0.3802, 0.4, 0.4004, 0.4247, 0.4497, 0.47, 0.4783,
                  0.48, 0.5929, 0.6797, 0.75, 0.7812, 0.8754, 0.88, 0.9,
                  1.037, 1.3, 1.363, 1.43, 1.53, 1.75, 1.965])
HZ_VAL = np.array([69.0, 69.0, 68.6, 83.0, 75.0, 75.0, 72.9, 77.0, 88.8,
                    83.0, 83.0, 95.0, 77.0, 87.1, 92.8, 89.0, 80.9,
                    97.0, 104.0, 92.0, 98.8, 105.0, 125.0, 90.0, 117.0,
                    154.0, 168.0, 160.0, 177.0, 140.0, 202.0, 186.5])
HZ_ERR = np.array([19.6, 12.0, 26.2, 8.0, 4.0, 5.0, 29.6, 14.0, 36.6,
                    14.0, 13.5, 17.0, 10.2, 11.2, 12.9, 49.6, 9.0,
                    62.0, 13.0, 8.0, 33.6, 12.0, 17.0, 40.0, 23.0,
                    20.0, 17.0, 33.6, 18.0, 14.0, 40.0, 50.4])

# EU and LCDM predictions for H(z) at CC redshifts
# Use exact CLASS LCDM background (consistent with §5, §7)
H_cc_lcdm = np.array([H_lcdm_func(z) for z in HZ_Z])
H_cc_eu = np.array([H_eu_interp(z) for z in HZ_Z])

chi2_cc_lcdm = np.sum(((HZ_VAL - H_cc_lcdm) / HZ_ERR)**2)
chi2_cc_eu = np.sum(((HZ_VAL - H_cc_eu) / HZ_ERR)**2)

print('=' * 60)
print('§9. COSMIC CHRONOMETERS H(z)')
print('=' * 60)
print(f'  Data: 32 points (Moresco+ 2022)')
print(f'  chi2 LCDM: {chi2_cc_lcdm:.1f} (chi2/N = {chi2_cc_lcdm/32:.2f})')
print(f'  chi2 EU:   {chi2_cc_eu:.1f} (chi2/N = {chi2_cc_eu/32:.2f})')
print(f'  Delta chi2: {chi2_cc_eu - chi2_cc_lcdm:+.1f}')

# Figure
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), height_ratios=[3, 1], sharex=True)

z_plot = np.linspace(0, 2.1, 200)
H_plot_l = [H_lcdm_func(z) for z in z_plot]
H_plot_e = [H_eu_interp(z) for z in z_plot]

ax1.errorbar(HZ_Z, HZ_VAL, HZ_ERR, fmt='o', ms=4, color='gray',
             alpha=0.6, capsize=2, label='CC data (Moresco+2022)')
ax1.plot(z_plot, H_plot_l, 'b-', lw=2, label=r'$\Lambda$CDM')
ax1.plot(z_plot, H_plot_e, 'r--', lw=2, label='EU')
ax1.set_ylabel('H(z) [km/s/Mpc]')
ax1.set_title('Cosmic Chronometers')
ax1.legend()

# Residuals
ax2.errorbar(HZ_Z, (HZ_VAL - H_cc_eu) / HZ_ERR, 1, fmt='rs', ms=4, alpha=0.6, label='EU')
ax2.axhline(0, color='gray', ls='--')
ax2.set_xlabel('z'); ax2.set_ylabel(r'$(H_{\rm obs} - H_{\rm EU})/\sigma$')

plt.tight_layout()
plt.savefig('figures/fig_NB04_CC.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] CC figure saved')


---
# §9.1. Posterior Predictive Distribution (PPD) Check

Statistical validation of Hubble diagram residuals:
Shapiro-Wilk normality test, Durbin-Watson serial correlation,
and Q-Q plot diagnostic.


In [ ]:
# ============================================================
# §9.1. PPD — Statistical Validation of SN Residuals
# ============================================================
from scipy import stats

print('=' * 60)
print('§9.1. POSTERIOR PREDICTIVE DISTRIBUTION CHECK')
print('=' * 60)

# Standardized residuals from the Pantheon+ fit (from §4c)
try:
    for label, residuals in [('EU', res_eu), ('LCDM', res_lcdm)]:
        std_res = residuals / mu_err_bin
        n_res = len(std_res)

        # Shapiro-Wilk normality test
        sw_stat, sw_p = stats.shapiro(std_res[:min(5000, n_res)])

        # Durbin-Watson serial correlation
        diffs = np.diff(std_res)
        dw = np.sum(diffs**2) / np.sum(std_res**2)

        # Runs test (sign changes)
        signs = np.sign(std_res)
        runs = 1 + np.sum(np.abs(np.diff(signs)) > 0)
        n_pos = np.sum(signs > 0)
        n_neg = np.sum(signs < 0)
        E_runs = 1 + 2*n_pos*n_neg / (n_pos + n_neg)

        print(f'\n  {label}:')
        print(f'    N residuals:        {n_res}')
        print(f'    Mean:               {np.mean(std_res):.3f} (expect ~0)')
        print(f'    Std:                {np.std(std_res):.3f} (expect ~1)')
        print(f'    Shapiro-Wilk:       W={sw_stat:.4f}, p={sw_p:.4f}')
        print(f'    Durbin-Watson:      DW={dw:.3f} (expect ~2.0)')
        print(f'    Runs:               {runs} (expected {E_runs:.0f})')
        status = 'PASS' if sw_p > 0.01 and 1.5 < dw < 2.5 else 'CHECK'
        print(f'    Status:             {status}')
except NameError:
    raise RuntimeError(
        'FATAL: SN residuals (res_eu, res_lcdm) not defined. '
        'Re-run §8 (Pantheon+) cell first.')


In [ ]:
# ============================================================
# §9.2. AGE OF THE UNIVERSE — Globular Cluster Constraint
# ============================================================
# The EU model must produce t₀ > 12.6 Gyr (Valcin+ 2021,
# arXiv:2102.04486) to avoid the "old stars problem".
# Extract from CLASS background: proper time [Gyr] at z=0.
# ============================================================

# Load proper time from CLASS background files
def extract_age(bg_data, col_map):
    """Extract universe age from CLASS background at z=0."""
    # Try both possible column naming conventions
    for col_name in ['proper time [Gyr]', '(.)proper time [Gyr]']:
        if col_name in col_map:
            z_arr = bg_data[:, col_map['z']]
            t_arr = bg_data[:, col_map[col_name]]
            idx_z0 = np.argmin(np.abs(z_arr))
            return t_arr[idx_z0]
    raise KeyError(
        f"FATAL: 'proper time [Gyr]' not found in CLASS columns. "
        f"Available: {list(col_map.keys())}")

t0_lcdm = extract_age(bg_lcdm, cols_l)
t0_eu   = extract_age(bg_eu, cols_e)

# Observational constraint: globular clusters
t_GC = 12.6  # Gyr, Valcin+ 2021 (95% CL lower bound)
t_GC_ref = "Valcin+ 2021, arXiv:2102.04486"

print('=' * 60)
print('§9.2. AGE OF THE UNIVERSE')
print('=' * 60)
print(f'  t₀(LCDM) = {t0_lcdm:.2f} Gyr')
print(f'  t₀(EU)   = {t0_eu:.2f} Gyr')
print(f'  t_GC     > {t_GC} Gyr ({t_GC_ref})')
print()
if t0_eu > t_GC:
    print(f'  ✅ t₀(EU) = {t0_eu:.2f} Gyr > {t_GC} Gyr — PASS')
else:
    print(f'  ⚠️  t₀(EU) = {t0_eu:.2f} Gyr < {t_GC} Gyr — FAIL')
if t0_lcdm > t_GC:
    print(f'  ✅ t₀(LCDM) = {t0_lcdm:.2f} Gyr > {t_GC} Gyr — PASS')
else:
    print(f'  ⚠️  t₀(LCDM) = {t0_lcdm:.2f} Gyr < {t_GC} Gyr — FAIL')


---
---
# Part V — The $H_0$ Tension

The central result of the EU framework: a two-scale resolution of the Hubble tension
via the Global Kinematic Imprint (GKI) and Local Kinematic Imprint (LKI).
We first prove the No-Go theorem (why Friedmann alone cannot reach $H_0 = 73$),
then present the virial mechanism and its observational evidence.


---
# §10. No-Go Theorem: Profile Likelihood

### The Geometric Wall

We demonstrate that **no late-time Friedmann modification** can resolve the
$H_0$ tension by computing the **profile likelihood** over $H_0 \in [60, 80]$.

For each $H_0$:
- **$\Lambda$CDM**: minimize $\chi^2$ over $\Omega_m$
- **EU-Friedmann**: evaluate at fixed $\varepsilon$, $z_{\rm trans}$ ($\Delta k = 0$)

Both profiles have their minimum at $H_0 \approx 68$ — the BAO + SN data
create a "geometric wall" that prevents any late-time model from reaching
$H_0 = 73$ within the Friedmann equation.

**The EU does not break through this wall — it goes around it** via the GKI/LKI mechanism:
the background Friedmann gives $H_0^{\rm bg} \approx 69.6$ (compatible with BAO),
and the virial channel (LKI) adds the local boost to $H_0^{\rm local} \approx 73.2$.


In [ ]:
# ============================================================
# §10. NO-GO THEOREM — Profile Likelihood Scan
# ============================================================
# Demonstrates that NO late-time Friedmann modification can
# reach H0 = 73 km/s/Mpc. Uses BAO + Cosmic Chronometers
# combined for maximum constraining power.
#
# Key insight: since EU conserves total energy density
# (CDM → DE transfer), E(z) ≈ E_LCDM(z). Both models
# hit the same "geometric wall" at H0 ≈ 68–69.
# GKI pushes EU to the wall (H0~69); LKI crosses it via sub-Friedmann local kinematics.
# ============================================================
from scipy.integrate import quad as _quad, solve_ivp
from scipy.interpolate import CubicSpline

print("=" * 60)
print("§10. NO-GO THEOREM — PROFILE LIKELIHOOD")
print("=" * 60)

c_km = 299792.458  # km/s

# LCDM Friedmann (exact)
def E_lcdm(z, Om): return np.sqrt(Om*(1+z)**3 + (1-Om))

# ── Physical constants (invariant under H0 scan) ──
omega_m_phys = Om_fid * (H0_fid / 100.0)**2  # ω_m = Ω_m h²
omega_c_phys = params['planck2018_observables']['omega_cdm']
assert omega_c_phys > 0.10 and omega_c_phys < 0.13, f"omega_cdm sanity: {omega_c_phys}"

# ── BAO chi2 for profile scan ──
# Use DESI DR2 data from §7b (already downloaded)
def chi2_bao_scan(H0, E_func):
    """BAO chi2 for profile scan. E_func(z) is dimensionless Hubble."""
    H_func = lambda z: H0 * E_func(z)
    th = np.zeros(N_desi)
    for k in range(N_desi):
        z = desi_z[k]
        DM, _ = _quad(lambda zp: c_km / H_func(zp), 0, z, limit=100)
        DH = c_km / H_func(z)
        rd_scan = rd_lcdm  # rd is pre-recombination, independent of late-time H0
        if desi_qty[k] == "DM_over_rs":
            th[k] = DM / rd_scan
        elif desi_qty[k] == "DH_over_rs":
            th[k] = DH / rd_scan
        elif desi_qty[k] == "DV_over_rs":
            DV = (z * DH * DM**2)**(1.0/3.0)
            th[k] = DV / rd_scan
    delta = desi_val - th
    return float(delta @ desi_cov_inv @ delta)

# ── CC chi2 ──
def chi2_cc_scan(H0, E_func):
    """Cosmic Chronometers chi2. E_func(z) is dimensionless Hubble."""
    chi2 = 0
    for j in range(len(HZ_Z)):
        H_pred = H0 * E_func(HZ_Z[j])
        chi2 += ((HZ_VAL[j] - H_pred) / HZ_ERR[j])**2
    return chi2

# ── Combined profile scan ──
H0_SCAN = np.linspace(60, 80, 200)

prof_lcdm = np.zeros(len(H0_SCAN))
prof_eu = np.zeros(len(H0_SCAN))

for i, H0_s in enumerate(H0_SCAN):
    h_s = H0_s / 100.0
    Om_s = omega_m_phys / h_s**2
    Oc_s = omega_c_phys / h_s**2
    Ob_nu_s = Om_s - Oc_s  # baryons + neutrinos
    OL_s = 1.0 - Om_s

    # LCDM: analytic E(z)
    E_lcdm_s = lambda z, Om=Om_s: np.sqrt(Om*(1+z)**3 + (1-Om))
    prof_lcdm[i] = chi2_cc_scan(H0_s, E_lcdm_s) + chi2_bao_scan(H0_s, E_lcdm_s)

    # EU: dynamic z_trans + survival fraction + ODE
    zt_s = (OL_s * (16*np.pi**2 - 1) / Om_s)**(1/3) - 1.0

    # epsilon(z) for this z_trans (default arg avoids late-binding)
    def eps_dyn(z, zt=zt_s):
        if z > zt: return 0.0
        return eps_IR / (1.0 + ((1+z)/(1+zt))**(1.0/b))

    # S(0): total CDM drain from z_trans to z=0
    I_tot, _ = _quad(lambda zp, zt=zt_s: eps_dyn(zp, zt)/(1+zp), 0, 200, limit=200)
    S_0_s = np.exp(-lam * I_tot)

    # Physical densities today (Friedmann closure: Ω_tot = 1 exact)
    Oc_today = Oc_s * S_0_s           # CDM after drain
    OL_today = 1.0 - (Oc_today + Ob_nu_s)  # DE absorbs the drain

    # ODE: integrate from z=0 to z=3 (past)
    # Variables: y[0] = Ω_cdm(z)/E²(z), y[1] = Ω_Λ(z)/E²(z)
    # But simpler: track actual density parameters
    def bg_sys(z, y, zt=zt_s):
        rc, rL = y  # CDM and DE density parameters
        eps = eps_dyn(z, zt)
        drc = (3.0 + lam * eps) * rc / (1+z)  # CDM: dilution + drain
        drL = -lam * eps * rc / (1+z)          # DE: absorbs drain
        return [drc, drL]

    z_eval = np.linspace(0, 3.0, 150)
    sol = solve_ivp(bg_sys, [0, 3.0], [Oc_today, OL_today],
                    t_eval=z_eval, rtol=1e-8, atol=1e-10)

    # E²(z) = Ω_cdm(z) + Ω_Λ(z) + Ω_{b+ν}(z)
    E2_z = sol.y[0] + sol.y[1] + Ob_nu_s * (1 + sol.t)**3
    E_spline = CubicSpline(sol.t, np.sqrt(np.maximum(E2_z, 0)))

    E_eu_dyn = lambda z, spl=E_spline: float(spl(z))  # default arg!
    prof_eu[i] = chi2_cc_scan(H0_s, E_eu_dyn) + chi2_bao_scan(H0_s, E_eu_dyn)

# ── Results ──
Dchi2_L = prof_lcdm - np.min(prof_lcdm)
Dchi2_E = prof_eu - np.min(prof_eu)

H0_best_L = H0_SCAN[np.argmin(prof_lcdm)]
H0_best_E = H0_SCAN[np.argmin(prof_eu)]

nogo_L = Dchi2_L[np.argmin(np.abs(H0_SCAN - H0_SH0ES))]
nogo_E = Dchi2_E[np.argmin(np.abs(H0_SCAN - H0_SH0ES))]

# CDM drain with lambda (verify against NB03 JSON)
f_drain_nogo = 1 - S_0_s  # from last iteration (H0=80), use fiducial below
I_fid, _ = _quad(lambda zp: epsilon_of_z(zp)/(1+zp), 0, 200, points=[z_trans], limit=200)
S_fid = np.exp(-lam * I_fid)
f_drain_fid = 1 - S_fid
print(f"  CDM drain fraction (fiducial, with λ): {f_drain_fid:.6f}")

print(f"  Data: BAO (DESI DR2, {N_desi} pts) + CC (Moresco+2022, 32 pts)")
print(f"  LCDM best-fit: H0 = {H0_best_L:.1f}, Dchi2 at H0_SH0ES = {nogo_L:.1f}")
print(f"  EU best-fit:   H0 = {H0_best_E:.1f}, Dchi2 at H0_SH0ES = {nogo_E:.1f}")
print(f"  -> Both models prefer H0 ~ 68 in Friedmann!")
print(f"  -> The \"geometric wall\" is at {np.sqrt(nogo_L):.1f} sigma (LCDM)")
print(f"  -> GKI reaches the wall (H0~69); LKI crosses via local kinematics (see §11): H0_EU={H0_EU:.2f}")

# Figure
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(H0_SCAN, Dchi2_L, "b-", lw=2.5, label=r"$\Lambda$CDM (BAO+CC)")
ax.plot(H0_SCAN, Dchi2_E, "r--", lw=2.5, label="EU Friedmann (BAO+CC)")
for lv, ls in [(1,r"$1\sigma$"), (4,r"$2\sigma$"), (9,r"$3\sigma$")]:
    ax.axhline(lv, color="gray", ls=":", lw=0.8, alpha=0.5)
    ax.text(80.2, lv, ls, fontsize=9, color="gray", va="center")
ax.axvline(H0_SH0ES, color="#FF6B00", ls="-", lw=2, alpha=0.8)
ax.axvspan(H0_SH0ES-0.86, H0_SH0ES+0.86, alpha=0.12, color="#FF6B00")
ax.text(73.4, max(nogo_L, nogo_E)*0.8, "SH0ES", fontsize=10,
        color="#FF6B00", fontweight="bold")
ax.annotate("GKI bypass", xy=(H0_EU, 0.5),
    xytext=(H0_best_E+1, nogo_E*0.4),
    arrowprops=dict(arrowstyle="->", color="green", lw=2),
    fontsize=11, color="green", fontweight="bold")
ax.set_xlabel(r"$H_0$ [km/s/Mpc]")
ax.set_ylabel(r"$\Delta\chi^2$")
ax.set_title("No-Go Theorem: Profile Likelihood (BAO + CC combined)")
ax.legend(); ax.set_xlim(60, 80.5)
ax.set_ylim(-0.5, max(nogo_L, nogo_E)*1.2)
plt.tight_layout()
plt.savefig("figures/fig_NB04_nogo.png", dpi=150, bbox_inches="tight")
plt.show()
print("[OK] No-Go figure saved")


---
# §11. H₀ Decomposition: GKI/LKI Channels

Two-layer analysis


In [ ]:
# ============================================================
# §11. H0 DECOMPOSITION — GKI / LKI (KBC Void)
# ============================================================
# The EU modification of H0 has two contributions:
#   GKI (Global Kinematic Imprint): background CDM evaporation → H0_CLASS_EU
#   LKI (Local Kinematic Imprint):  KBC Void outflow (Wu & Huterer 2017)
#
# H0_local = H0_GKI + dH0_void    [universal: all local calibrators]
# ============================================================

# GKI: adiabatic channel (background only)
if USE_MCMC_PARAMS:
    H0_GKI = C2_DERIVED['H0']  # 68.886 from MCMC C2 (auto-consistent)
    print(f'  [MODE B] H0_GKI = {H0_GKI:.3f} (MCMC C2)')
else:
    H0_GKI = nb03['CLASS_background']['H0_EU_kmsMpc']  # 68.56 (Planck fiducial)

# LKI: KBC Void outflow (from NB02, Wu & Huterer 2017)
dH0_void = nb02['lki_void']['dH0_void']
H0_local = nb02['lki_void']['H0_local']  # = H0_GKI + dH0_void

# Verify consistency
assert abs(H0_local - (H0_GKI + dH0_void)) < 0.5, \
    f"H0_local inconsistency: {H0_local:.2f} vs {H0_GKI + dH0_void:.2f}"

# GKI and LKI contributions
GKI_val = H0_GKI - H0_Planck
LKI_val = dH0_void

print('=' * 60)
print('§11. H0 DECOMPOSITION (KBC Void)')
print('=' * 60)
print(f'  H0(Planck)           = {H0_Planck:.2f} km/s/Mpc')
print(f'  H0(GKI only)         = {H0_GKI:.2f} km/s/Mpc  [= H0_CLASS_EU]')
print(f'  dH0(KBC Void)        = +{dH0_void:.2f} km/s/Mpc  [Wu & Huterer 2017]')
print(f'  H0(EU local)         = {H0_local:.2f} km/s/Mpc  [universal]')
print(f'  H0(SH0ES)            = {H0_SH0ES} ± {H0_SH0ES_err}')
print()
print(f'  GKI channel: +{GKI_val:.2f} km/s/Mpc (background CDM drain)')
print(f'  LKI channel: +{LKI_val:.2f} km/s/Mpc (KBC Void outflow)')
print(f'  Total:       +{GKI_val + LKI_val:.2f} km/s/Mpc')

# ── Figure: H0 hierarchy ──
fig, ax = plt.subplots(figsize=(10, 5))

labels = ['Planck 2018\n(CMB)', 'EU (GKI)', 'EU (Local)\n=GKI+Void', 'SH0ES 2024\n(Cepheids)']
values = [H0_Planck, H0_GKI, H0_local, H0_SH0ES]
errors = [0.54, 0, 0, H0_SH0ES_err]
colors = ['#1f77b4', '#ff7f0e', '#d62728', '#2ca02c']

ax.barh(labels, values, xerr=errors, color=colors, alpha=0.8, height=0.5,
        capsize=4)
ax.set_xlabel(r'$H_0$ [km/s/Mpc]')
ax.set_title(r'$H_0$ Tension: EU Resolution (KBC Void)')
ax.axvline(H0_SH0ES, color='green', ls=':', alpha=0.3)
ax.axvline(H0_Planck, color='blue', ls=':', alpha=0.3)
ax.set_xlim(65, 76)
for i, (v, l) in enumerate(zip(values, labels)):
    ax.text(v + 0.2, i, f'{v:.2f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('figures/fig_NB04_H0.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/fig_NB04_H0.pdf', bbox_inches='tight')
plt.show()
print('[OK] H0 figure saved')


---
# §11.1. Virial Mechanism: Adiabatic Relaxation

The CDM mass loss $f_{\rm drain} \approx 8\%$ drives an adiabatic virial
relaxation of gravitationally bound systems. By the virial theorem ($2K + U = 0$):

- **Virial radius**: $R \to R / (1 - f_{\rm drain})$ (expands)
- **Orbital velocity**: $v \to v \times (1 - f_{\rm drain})$ (slows)

The expansion velocity field $v_{\rm expand} \propto r$ **mimics the Hubble law**,
creating an additional blue-shift that SH0ES interprets as a higher $H_0$.

This is the **Local Kinematic Imprint (LKI)** — a sub-Friedmann mechanism
invisible to BAO/SN probes (which measure integrated distances, not local velocities).


In [ ]:
# ============================================================
# §11.1. VIRIAL MECHANISM — Historical Note
# ============================================================
# PHY-3: Halo adiabatic response REMOVED from the model.
# ΔH₀ ~ 0.009 km/s/Mpc (irrisory vs void ΔH₀ ~ 3.6).
# LKI = KBC Void outflow only (Wu & Huterer 2017).
# This section is kept for completeness only.
# ============================================================

# CDM drain from NB03 JSON (validated)
fcdm = nb03['bianchi']['fcdm_z0']
f_drain = nb03['CLASS_background']['cdm_drain_pct'] / 100.0

print('=' * 60)
print('§11.1. VIRIAL MECHANISM — HISTORICAL NOTE')
print('=' * 60)
print(f'  CDM survival fraction: f_cdm = {fcdm:.6f}')
print(f'  CDM drain: {f_drain*100:.2f}%')
print()
print('  ⚠️  PHY-3 (NB02): Halo adiabatic response removed.')
print('  The virial mechanism produces ΔH₀ ~ 0.009 km/s/Mpc,')
print('  which is negligible compared to the KBC Void outflow')
print(f'  ΔH₀_void = {nb02["lki_void"]["dH0_void"]:.2f} km/s/Mpc.')
print()
print('  ✅ LKI = KBC Void outflow (universal, all local calibrators)')
print('  ✅ No disk/halo distinction needed')


---
# §11.2. H₀ Two-Scale Compilation

10 independent $H_0$ measurements classified by LKI sensitivity:

- **Disk calibrators** (Cepheids, Miras): inside galactic potential wells →
  full LKI exposure → expect $H_0 \approx H_0^{\rm local}$
- **Halo calibrators** (TRGB, JAGB): galaxy halos/outskirts →
  LKI attenuated → expect $H_0 \approx H_0^{\rm bg}$
- **Geometric** (lensing, masers): full line-of-sight velocity →
  expect $H_0 \approx H_0^{\rm local}$
- **Early Universe** (CMB): ΛCDM projection from $z \approx 1100$


In [ ]:
# ============================================================
# §11.2. OBSERVATIONAL CONSISTENCY — Universal H0 Hierarchy
# ============================================================
# After KBC Void (U6/U13): ALL local measurements see the
# same H0_local = H0_GKI + dH0_void ≈ 72.49 km/s/Mpc.
# The difference between SH0ES and TRGB is Rung 2 calibration,
# NOT a disk/halo LKI distinction.
# ============================================================

# Import observational H0 from NB02 JSON
H0_TRGB = nb02['hierarchy']['TRGB_JWST']['H0_obs']
err_TRGB = nb02['hierarchy']['TRGB_JWST']['err']
H0_JAGB = nb02['hierarchy']['JAGB_JWST']['H0_obs']
err_JAGB = nb02['hierarchy']['JAGB_JWST']['err']
H0_TRGB_mixed = nb02['hierarchy']['TRGB_mixed']['H0_obs']
err_TRGB_mixed = nb02['hierarchy']['TRGB_mixed']['err']

# Universal EU prediction for all local calibrators
H0_EU_universal = nb02['lki_void']['H0_local']

# Observational hierarchy (NB02 §5 style)
obs_hierarchy = [
    ('CMB/Planck 2018',    H0_Planck,     0.54,            H0_GKI,          'Input (CMB)'),
    ('JAGB/JWST',          H0_JAGB,       err_JAGB,        H0_EU_universal, 'GKI + void'),
    ('TRGB/JWST',          H0_TRGB,       err_TRGB,        H0_EU_universal, 'GKI + void'),
    ('TRGB (mixed)',       H0_TRGB_mixed, err_TRGB_mixed,  H0_EU_universal, 'GKI + void'),
    ('SH0ES 2024',         H0_SH0ES,      H0_SH0ES_err,    H0_EU_universal, 'GKI + void'),
]

print('=' * 60)
print('§11.2. OBSERVATIONAL CONSISTENCY — Universal Hierarchy')
print('=' * 60)
print(f'  EU H0_local (universal) = {H0_EU_universal:.2f} km/s/Mpc')
print(f'  (= H0_GKI + dH0_void = {H0_GKI:.2f} + {dH0_void:.2f})')
print()
print(f'  {"Method":>22s}  {"H0_obs":>6s} {"±err":>5s}  {"H0_EU":>6s}  {"Δσ":>5s}  Channel')
print('  ' + '=' * 65)

chi2_hierarchy = 0.0
for name, h0_obs, err, h0_eu, channel in obs_hierarchy:
    tension = abs(h0_obs - h0_eu) / err
    chi2_hierarchy += ((h0_obs - h0_eu) / err)**2
    print(f'  {name:>22s}  {h0_obs:6.2f} {err:5.2f}  {h0_eu:6.2f}  {tension:5.2f}σ  {channel}')

N_obs = len(obs_hierarchy)
print(f'\n  χ² (EU universal) = {chi2_hierarchy:.1f} / {N_obs} obs')
print(f'  χ²/N = {chi2_hierarchy/N_obs:.2f}')

# ── Bar chart: Observational Hierarchy (NB02 style) ──
fig, ax = plt.subplots(figsize=(12, 6))

names = [o[0] for o in obs_hierarchy]
h0_obs_arr = [o[1] for o in obs_hierarchy]
err_arr = [o[2] for o in obs_hierarchy]
h0_eu_arr = [o[3] for o in obs_hierarchy]

x = np.arange(len(names))
width = 0.35

bars_obs = ax.bar(x - width/2, h0_obs_arr, width, yerr=err_arr,
                  label='Observed', color='#3498db', alpha=0.8,
                  capsize=5, edgecolor='#2c3e50', linewidth=1.5)
bars_eu = ax.bar(x + width/2, h0_eu_arr, width,
                 label='EU prediction', color='#e74c3c', alpha=0.8,
                 edgecolor='#c0392b', linewidth=1.5)

ax.axhline(H0_Planck, color='gray', ls=':', lw=1.5, alpha=0.5, label=f'Planck ({H0_Planck})')
ax.axhline(H0_EU_universal, color='#e74c3c', ls='--', lw=1.5, alpha=0.5,
           label=f'EU universal ({H0_EU_universal:.2f})')

for i, (o, e, p) in enumerate(zip(h0_obs_arr, err_arr, h0_eu_arr)):
    ax.text(i - width/2, o + e + 0.3, f'{o:.1f}', ha='center', fontsize=9)
    ax.text(i + width/2, p + 0.3, f'{p:.1f}', ha='center', fontsize=9, color='#c0392b')

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylabel(r'$H_0$ [km/s/Mpc]')
ax.set_title('Observational Consistency: Universal EU Hierarchy')
ax.legend(loc='lower right', fontsize=10)
ax.set_ylim(62, 78)
plt.tight_layout()
plt.savefig('figures/fig_NB04_H0_hierarchy.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] Hierarchy figure saved')


---
# §11.3. LKI Fraction Analysis

For each measurement: $H_0^{\rm obs} = H_0^{\rm bg} + f_{\rm LKI} \times \text{LKI}$

- $f_{\rm LKI} = 0$ → sees background only (halo calibrators)
- $f_{\rm LKI} = 1$ → sees full local velocity (disk calibrators)

The SH0ES–TRGB split is **not a systematic error** — it is the LKI signature.


### §11.3. $f_{\rm LKI}$ Fraction Analysis

Decompose the local Hubble boost into disk-scale and halo-scale contributions.
The $f_{\rm LKI}$ parameter quantifies how much of the observed $H_0$ tension
comes from structures at different scales (virial radii).

- $f_{\rm disk} \approx 1$: bulk of the LKI boost is at galactic disk scales
- $f_{\rm halo} \ll 1$: negligible contribution from halo/cluster scales

In [ ]:
# ============================================================
# §11.3. [REMOVED] — f_LKI Fraction Analysis
# ============================================================
# This section was removed in the structural refactoring (U13).
# Reason: The two-scale (disk/halo) model is obsolete.
# With KBC Void, ALL local calibrators see the same H0_local.
# See §11.2 for the universal observational hierarchy.
# ============================================================
print('§11.3. [REMOVED] — Two-Scale f_LKI analysis replaced by §11.2 universal hierarchy')


---
# §11.4. $\chi^2$ Two-Scale vs ΛCDM

Compare single-scale ΛCDM ($H_0 = 67.36$) against EU two-scale
($H_0^{\rm bg}$ for halo/CMB, $H_0^{\rm local}$ for disk/geometric)
using the 10-method compilation.


In [ ]:
# ============================================================
# §11.4. [REMOVED] — Chi2 Two-Scale Comparison
# ============================================================
# This section was removed in the structural refactoring (U13).
# Replaced by the universal hierarchy chi2 in §11.2.
# ============================================================
print('§11.4. [REMOVED] — Two-Scale chi2 replaced by §11.2 universal hierarchy')


---
# §11.5. CLASS Exact Verification

Cross-check the analytic GKI prediction against the full CLASS-EU
background solution. The CDM survival fraction $f_{\rm cdm}$ and
$H_0$ ratio must agree to machine precision.


In [ ]:
# TKI_GKI: compute from H0_GKI / H0_Planck (GKI only, no void)
# H0_GKI = H0_Planck * exp(TKI_GKI) → TKI_GKI = ln(H0_GKI / H0_Planck)
TKI_GKI = np.log(nb02['gki']['H0_GKI'] / H0_Planck)

# ============================================================
# §11.5. CLASS EXACT VERIFICATION
# ============================================================

# CDM density fraction from CLASS output (exact)
rho_cdm_eu = bg_eu[:, cols_e['(.)rho_cdm']]
rho_cdm_lcdm = bg_lcdm[:, cols_l['(.)rho_cdm']]

idx_z0_e = np.argmin(np.abs(z_e))
idx_z0_l = np.argmin(np.abs(z_l))

fcdm_class = rho_cdm_eu[idx_z0_e] / rho_cdm_lcdm[idx_z0_l]

# H0 ratio from CLASS
H0_class_eu = H_eu_interp(0.0)  # km/s/Mpc
H0_ratio_class = H0_class_eu / H0_Planck
H0_ratio_analytic = np.exp(TKI_GKI)

print('=' * 60)
print('§11.5. CLASS EXACT VERIFICATION')
print('=' * 60)
print(f'  f_cdm(z=0):')
print(f'    CLASS:    {fcdm_class:.6f}')
print(f'    Analytic: {np.exp(-TKI_GKI * Omega_m / (Omega_m + Omega_L)):.6f}')
print(f'  H0 ratio (EU/Planck):')
print(f'    CLASS:    {H0_ratio_class:.6f}')
print(f'    Analytic: {H0_ratio_analytic:.6f}')
print(f'    Agreement: {abs(H0_ratio_class - H0_ratio_analytic)/H0_ratio_analytic*100:.4f}%')


---
---
# Part VI — Statistical Assessment

Model comparison, forecasts, and honest assessment of the framework's limitations.


---
# §12. Model Selection

Information criteria comparison with $\Delta k = 0$ (zero extra free parameters).
The EU model is a **prediction**, not a fit — all parameters are UV-derived.


In [ ]:
# ============================================================
# §12. MODEL SELECTION — chi2, AIC, BIC
# ============================================================
# EU has Delta_k = 0 extra parameters relative to LCDM.
# Therefore:
#   Delta_AIC = Delta_chi2 + 2*Delta_k = Delta_chi2
#   Delta_BIC = Delta_chi2 + Delta_k * ln(N) = Delta_chi2
# ============================================================

# Recalculate tensions with latest (growth-corrected) values
S8_tension_lcdm = abs(S8_lcdm - S8_obs) / S8_obs_err
S8_tension_eu = abs(S8_eu - S8_obs) / S8_obs_err

print('=' * 60)
print('§12. MODEL SELECTION CRITERIA')
print('=' * 60)

# ── Tension summary ──
H0_tension_lcdm = abs(H0_Planck - H0_SH0ES) / H0_SH0ES_err
H0_tension_eu = abs(H0_local - H0_SH0ES) / H0_SH0ES_err  # Formula B: use two-step H0_local

print(f'\n  H0 tension:')
print(f'    LCDM: {H0_tension_lcdm:.1f} sigma')
print(f'    EU:   {H0_tension_eu:.2f} sigma')

print(f'\n  S8 tension:')
print(f'    LCDM: {S8_tension_lcdm:.1f} sigma')
print(f'    EU:   {S8_tension_eu:.1f} sigma')

if chi2_lcdm is not None:
    print(f'\n  CMB chi2 (vs Planck 2018 TT):')
    print(f'    LCDM: {chi2_lcdm:.1f}')
    print(f'    EU:   {chi2_eu:.1f}')
    print(f'    Delta chi2: {delta_chi2:+.1f}')
    print(f'    Delta AIC:  {delta_AIC:+.1f} (Delta_k = 0)')

print(f'\n  Extra free parameters (Delta_k): 0')
print(f'  EU is NOT a fit — it is a PREDICTION.')

# ── Summary table ──
print('\n' + '=' * 60)
print('COMPREHENSIVE TENSION TABLE')
print('=' * 60)
print(f'{"Probe":<25} {"LCDM":>12} {"EU":>12} {"Status":>10}')
print('-' * 60)
print(f'{"H0 [km/s/Mpc]":<25} {H0_Planck:>12.2f} {H0_local:>12.2f} {"":>10}')
print(f'{"H0 tension [sigma]":<25} {H0_tension_lcdm:>12.1f} {H0_tension_eu:>12.2f} {"RESOLVED":>10}')
print(f'{"sigma8":<25} {sigma8_lcdm:>12.4f} {sigma8_eu:>12.4f} {"":>10}')
print(f'{"S8":<25} {S8_lcdm:>12.4f} {S8_eu:>12.4f} {"":>10}')
print(f'{"S8 tension [sigma]":<25} {S8_tension_lcdm:>12.1f} {S8_tension_eu:>12.1f} {"REDUCED" if S8_tension_eu < S8_tension_lcdm else "":>10}')
if chi2_lcdm is not None:
    print(f'{"Delta chi2 (CMB)":<25} {0:>12.1f} {delta_chi2:>12.1f} {"":>10}')
print(f'{"Extra params (Delta_k)":<25} {0:>12d} {0:>12d} {"ZERO":>10}')
print(f'{"f_cdm(z=0)":<25} {1.0:>12.4f} {nb03["bianchi"]["fcdm_z0"]:>12.4f} {"":>10}')
print(f'{"Bianchi violation":<25} {0:>12.1e} {nb03["bianchi"]["energy_violation"]:>12.1e} {"PASS":>10}')


---
# §12.1. DESI/Euclid Forecasts

Projected distinguishability of the EU equation of state deviation
$\Delta w = 1 + w_0$ with future survey sensitivities.


In [ ]:
# ============================================================
# §12.1. DESI/EUCLID DISTINGUISHABILITY FORECAST
# ============================================================

# EU equation of state deviation
w0_eu = -1.0 + (eps_IR / 3.0) * (Omega_m / Omega_L)
delta_w = abs(1.0 + w0_eu)

surveys = {
    'Current (Planck+BAO)':    {'sigma_w': 0.05, 'color': '#FF5722'},
    'DESI Full (2029)':        {'sigma_w': 0.02, 'color': '#2196F3'},
    'Euclid (2030)':           {'sigma_w': 0.015, 'color': '#4CAF50'},
    'DESI+Euclid':             {'sigma_w': 0.010, 'color': '#9C27B0'},
    'CMB-S4+DESI+Euclid':     {'sigma_w': 0.005, 'color': '#FF9800'},
}

print('=' * 60)
print('§12.1. FORECAST — EU DISTINGUISHABILITY')
print('=' * 60)
print(f'  EU w0 = {w0_eu:.6f}')
print(f'  |Delta w| = |1 + w0| = {delta_w:.6f}')
print(f'  {"Survey":<25} {"sigma_w":<10} {"Significance":<12}')
print('  ' + '-' * 48)

names = list(surveys.keys())
sigmas = []
for name, s in surveys.items():
    sig = delta_w / s['sigma_w']
    sigmas.append(sig)
    print(f'  {name:<25} {s["sigma_w"]:<10.3f} {sig:<12.1f} sigma')

# Figure
fig, ax = plt.subplots(figsize=(10, 6))
colors = [s['color'] for s in surveys.values()]
bars = ax.bar(names, sigmas, color=colors, edgecolor='black', alpha=0.85, width=0.6)
ax.axhline(3, color='darkred', ls='--', lw=2, label=r'$3\sigma$ (evidence)')
ax.axhline(5, color='darkblue', ls='--', lw=2, label=r'$5\sigma$ (discovery)')
for bar, s in zip(bars, sigmas):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
            f'{s:.1f}σ', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel(r'Significance ($|\Delta w|/\sigma_w$)')
ax.set_title(f'EU Distinguishability: $|\\Delta w| = {delta_w:.4f}$')
ax.legend(); plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('figures/fig_NB04_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] Forecast figure saved')


---
# §12.2. Honest Limitations & Caveats

> **Scientific integrity requires explicit acknowledgment of what this
> analysis does and does not demonstrate.**

1. **$f_{\rm LKI}$ assignment is model-based, not first-principles.**
   We classify methods as "disk" or "halo" based on calibrator environment.
   A rigorous derivation of $f_{\rm LKI}(r)$ requires N-body simulations
   with EU-modified gravity.

2. **The disk/halo classification is binary; reality is continuous.**
   In practice $f_{\rm LKI}$ depends on calibrator position, host mass,
   and survey selection. Our binary scheme is a zeroth-order approximation.

3. **Some $H_0$ measurements are not fully independent.**
   SH0ES and SH0ES+JWST share calibration data. TRGB and CCHP use
   related anchors. The $\chi^2$ treats all 10 as independent.

4. **The $\chi^2$ comparison with ΛCDM is illustrative.**
   A rigorous model comparison requires the full posterior from MCMC
   (NB04 Cobaya). The $\chi^2$ here demonstrates *qualitative*
   superiority, not *definitive* Bayesian evidence.


---
---
# Part VII — Export

Save all validated results and references.


---
# §13. Summary & Export

Export all validated observational results for downstream use.


---
# §14. References

## Observational Data
1. Planck Collaboration (2020). *Planck 2018 results. VI.*
   A&A **641**, A6. [arXiv:1807.06209](https://arxiv.org/abs/1807.06209)
2. Breuval, L., Riess, A. G. et al. (2024). *SMC Cepheids: New anchor for SH0ES.*
   ApJ **973**, 30. [arXiv:2404.08038](https://arxiv.org/abs/2404.08038)
3. Abbott, T. M. C. et al. (DES Collaboration, 2022). *DES Year 3: Cosmic shear.*
   PRD **105**, 023520. [arXiv:2105.13549](https://arxiv.org/abs/2105.13549)

## Galaxy Surveys & Full-Shape Clustering
4. DESI Collaboration (2024). *DESI DR1 BAO measurements.*
   [arXiv:2411.12021](https://arxiv.org/abs/2411.12021)
5. DESI Collaboration (2024). *DESI DR1 Full-Shape analysis: ShapeFit.*
   [arXiv:2411.12022](https://arxiv.org/abs/2411.12022)
6. Brieden, S., Gil-Marín, H. & Verde, L. (2021). *ShapeFit: Extracting the
   power spectrum shape information.* JCAP **12**, 054.
   [arXiv:2106.11930](https://arxiv.org/abs/2106.11930)

## Type Ia Supernovae
7. Scolnic, D. et al. (2022). *The Pantheon+ sample.*
   ApJ **938**, 113. [arXiv:2112.03863](https://arxiv.org/abs/2112.03863)
8. Brout, D. et al. (2022). *Pantheon+ analysis.*
   ApJ **938**, 110. [arXiv:2202.04077](https://arxiv.org/abs/2202.04077)

## Cosmic Chronometers
9. Moresco, M. et al. (2022). *Setting the stage for cosmic chronometers.*
   Living Rev. Rel. **25**, 6. [arXiv:2201.07241](https://arxiv.org/abs/2201.07241)

## Weak Lensing
10. Heymans, C. et al. (2021). *KiDS-1000 cosmic shear.* A&A **646**, A140.
    [arXiv:2007.15632](https://arxiv.org/abs/2007.15632)
11. Wright, A. H. et al. (2025). *KiDS-Legacy: Cosmic shear.*
    A&A **703**, A158. [arXiv:2503.19441](https://arxiv.org/abs/2503.19441)

## Perturbation Theory
12. Valiviita, J., Majerotto, E. & Maartens, R. (2008). *Large-scale instability in
    IDE fluids.* JCAP **07**, 020. [arXiv:0804.0232](https://arxiv.org/abs/0804.0232)


In [ ]:
# ============================================================
# §12. GRAND SUMMARY — Tension Table & Model Selection
# ============================================================
# Final "Money Shot": all observables in one table.
# All values are COMPUTED, not hardcoded.
# FAIL-FAST: NO try/except. If a variable is missing,
# the notebook must crash to alert the user.
# ============================================================

print('=' * 70)
print('§12. GRAND SUMMARY — Tension Table')
print('=' * 70)

# ── Collect all chi2 values (FAIL-FAST: no try/except) ──
grand_chi2 = {
    'CMB_TT':    {'LCDM': chi2_lcdm,     'EU': chi2_eu,       'dof': ndof},
    'BAO_DR2':   {'LCDM': chi2_bao_lcdm, 'EU': chi2_bao_eu,   'dof': N_desi},
    # ShapeFit DR1 excluded: correlated with BAO DR2 (same DESI tracers)
    'Pantheon+': {'LCDM': chi2_sn_lcdm,  'EU': chi2_sn_eu,    'dof': N_sn},
    'CC':        {'LCDM': chi2_cc_lcdm,  'EU': chi2_cc_eu,    'dof': len(HZ_Z)},
}

# ── Grand chi2 ──
chi2_grand_lcdm = sum(v['LCDM'] for v in grand_chi2.values())
chi2_grand_eu   = sum(v['EU'] for v in grand_chi2.values())
dof_total       = sum(v['dof'] for v in grand_chi2.values())

print(f'\n  {"Dataset":>15s}  {"chi2(LCDM)":>10s}  {"chi2(EU)":>10s}  {"D_chi2":>8s}  {"dof":>5s}')
print(f'  {"-"*15}  {"-"*10}  {"-"*10}  {"-"*8}  {"-"*5}')
for name, vals in grand_chi2.items():
    dchi2 = vals['EU'] - vals['LCDM']
    print(f'  {name:>15s}  {vals["LCDM"]:10.2f}  {vals["EU"]:10.2f}  {dchi2:+8.2f}  {vals["dof"]:5d}')
print(f'  {"-"*15}  {"-"*10}  {"-"*10}  {"-"*8}  {"-"*5}')
print(f'  {"GRAND TOTAL":>15s}  {chi2_grand_lcdm:10.2f}  {chi2_grand_eu:10.2f}  {chi2_grand_eu - chi2_grand_lcdm:+8.2f}  {dof_total:5d}')

# ── Tension table ──
print(f'\n{"="*70}')
print('TENSION SUMMARY')
print(f'{"="*70}')
print(f'  {"Observable":>20s}  {"Obs +/- err":>15s}  {"LCDM":>8s}  {"EU":>8s}  {"s(LCDM)":>8s}  {"s(EU)":>6s}')
print(f'  {"-"*20}  {"-"*15}  {"-"*8}  {"-"*8}  {"-"*8}  {"-"*6}')

tension_rows = [
    ('H0 (SH0ES)',     H0_SH0ES,  H0_SH0ES_err,  H0_Planck, H0_local),
    ('H0 (TRGB/JWST)', H0_TRGB,   err_TRGB,       H0_Planck, H0_local),
    ('H0 (JAGB/JWST)', H0_JAGB,   err_JAGB,       H0_Planck, H0_local),
    ('S8 (DES-Y3)',    S8_obs,     S8_obs_err,     S8_lcdm,   S8_eu),
    ('t0 [Gyr]',       t_GC,      0.40,            t0_lcdm,   t0_eu),
]

for name, obs, err, pred_l, pred_e in tension_rows:
    sig_l = abs(obs - pred_l) / err
    sig_e = abs(obs - pred_e) / err
    print(f'  {name:>20s}  {obs:7.2f} +/- {err:<5.2f}  {pred_l:8.2f}  {pred_e:8.2f}  {sig_l:7.1f}s  {sig_e:5.1f}s')

fav = 'EU' if chi2_grand_eu < chi2_grand_lcdm else 'LCDM'
print(f'\n  > Grand chi2: EU {"better" if fav == "EU" else "worse"} by D_chi2 = {chi2_grand_eu - chi2_grand_lcdm:+.1f}')
print(f'  > EU resolves H0 tension: {abs(H0_SH0ES - H0_local)/H0_SH0ES_err:.1f}s (vs {abs(H0_SH0ES - H0_Planck)/H0_SH0ES_err:.1f}s in LCDM)')
print(f'  > EU resolves S8 tension: {abs(S8_obs - S8_eu)/S8_obs_err:.1f}s (vs {abs(S8_obs - S8_lcdm)/S8_obs_err:.1f}s in LCDM)')


In [ ]:
# ============================================================
# §13. EXPORT — Save NB04 results
# ============================================================
import json as _json

print('=' * 60)
print('§13. EXPORT — NB04 Results')
print('=' * 60)

# ── Collect all results ──
nb04_results = {
    'metadata': {
        'notebook': 'NB04_Observational_Validation',
        'version': 'v1.0',
        'date': str(pd.Timestamp.now()) if 'pd' in dir() else 'N/A',
        'description': 'Observational validation of EU vs LCDM',
        'upstream': ['NB01_params.json', 'NB02_results.json', f'NB03_{MODE}_results.json'],
        'mode': MODE,
        'mode_label': MODE_LABEL,
        'mode_description': (
            'Mode A: Planck LCDM fiducial in CLASS-EU. '
            'Mode B: MCMC C2 posteriors in CLASS-EU. '
            'LCDM reference identical in both.'
        ),
    },
    'h0_decomposition': {
        'H0_Planck': H0_Planck,
        'H0_GKI': H0_GKI,
        'dH0_void': dH0_void,
        'H0_local': H0_local,
        'H0_SH0ES': H0_SH0ES,
        'H0_SH0ES_err': H0_SH0ES_err,
        'GKI_contribution': GKI_val,
        'LKI_contribution': LKI_val,
    },
    'observational_hierarchy': {
        'method': 'KBC Void (Wu & Huterer 2017)',
        'H0_EU_universal': H0_local,
    },
    'bridge': {
        'sigma8_lcdm': sigma8_lcdm,
        'sigma8_eu': sigma8_eu,
        'S8_lcdm': S8_lcdm,
        'S8_eu': S8_eu,
        'S8_obs': S8_obs,
        'S8_obs_err': S8_obs_err,
        'Omega_m_lcdm': Omega_m_lcdm,
        'Omega_m_eu': Omega_m_eu,
        'rd_lcdm': rd_lcdm,
        'rd_eu': rd_eu,
    },
}

# BAO
nb04_results['bao_dr2'] = {
    'chi2_lcdm': chi2_bao_lcdm,
    'chi2_eu': chi2_bao_eu,
    'delta_chi2': chi2_bao_eu - chi2_bao_lcdm,
    'N_data': int(N_desi),
}

# ShapeFit
nb04_results['shapefit_dr1'] = {
    'chi2_lcdm': chi2_lc_tot,
    'chi2_eu': chi2_eu_tot,
    'delta_chi2': chi2_eu_tot - chi2_lc_tot,
    'n_bins': len(desi_bins),
    'n_dof': n_dof,
    'growth_suppression': growth_suppression,
}

# No-Go
nb04_results['nogo'] = {
    'H0_best_lcdm': float(H0_best_L),
    'H0_best_eu': float(H0_best_E),
    'Dchi2_at_73': float(nogo_L),
    'sigma_wall': float(np.sqrt(nogo_L)),
}

# Age (if available)
nb04_results['age'] = {
    't0_lcdm_Gyr': t0_lcdm,
    't0_eu_Gyr': t0_eu,
    't_GC_Gyr': t_GC,
    'pass': bool(t0_eu > t_GC),
}

# Grand chi2
nb04_results['grand_chi2'] = {
    'chi2_lcdm_total': chi2_grand_lcdm,
    'chi2_eu_total': chi2_grand_eu,
    'delta_chi2': chi2_grand_eu - chi2_grand_lcdm,
    'dof_total': dof_total,
    'datasets': list(grand_chi2.keys()),
}

# Tensions
H0_tension_lcdm = abs(H0_SH0ES - H0_Planck) / H0_SH0ES_err
H0_tension_eu = abs(H0_SH0ES - H0_local) / H0_SH0ES_err
S8_tension_lcdm = abs(S8_lcdm - S8_obs) / S8_obs_err
S8_tension_eu = abs(S8_eu - S8_obs) / S8_obs_err

nb04_results['tensions'] = {
    'H0_SH0ES_sigma_lcdm': H0_tension_lcdm,
    'H0_SH0ES_sigma_eu': H0_tension_eu,
    'S8_DES_sigma_lcdm': S8_tension_lcdm,
    'S8_DES_sigma_eu': S8_tension_eu,
}

# Save
output_path = f'NB04_{MODE}_results.json'
with open(output_path, 'w') as f:
    _json.dump(nb04_results, f, indent=2, default=float)
print(f'  [SAVED] {output_path}  (Mode {MODE})')
print(f'  Keys: {list(nb04_results.keys())}')
print()
print(f'  H0 tension: {H0_tension_lcdm:.1f}σ (ΛCDM) → {H0_tension_eu:.2f}σ (EU)')
print(f'  S8 tension: {S8_tension_lcdm:.1f}σ (ΛCDM) → {S8_tension_eu:.1f}σ (EU)')


In [ ]:
# ============================================================
# §15. MODE B — Automatic Re-run (if C2 available)
# ============================================================

if DUAL_MODE:
    print('=' * 60)
    print('§15. MODE B — Automatic Re-run with MCMC C2 parameters')
    print('=' * 60)

    # ── Load C2 ──
    _c2_path_B = next(p for p in _c2_paths if os.path.exists(p))
    with open(_c2_path_B) as _f:
        c2_B = json.load(_f)

    _cp_B = c2_B['cosmological_params']
    _dp_B = c2_B['derived_params']

    # ── Save Mode A tensions for comparison ──
    MODE_A_tensions = dict(nb04_results.get('tensions', {}))

    # ── C2 values ──
    MODE = 'B'
    MODE_LABEL = 'MCMC C2 (paper-ready)'

    H0_GKI_B     = _dp_B['H0']['mean']
    sigma8_eu_B  = _dp_B['sigma8']['mean']
    S8_eu_B      = _dp_B['S8']['mean']
    Omega_m_eu_B = _dp_B['Omega_m']['mean']
    rd_eu_B      = _dp_B['rdrag']['mean']
    H0_LKI_B     = _dp_B['H0_LKI']['mean']

    # ── LKI void boost (UV prediction from NB02, scaled to Mode B H0) ──
    # The void boost is a MODEL PREDICTION (NB02 UV derivation),
    # not a quantity fitted by MCMC. Mode B only adjusts the 6 Planck
    # params to remove LCDM bias; the void physics is unchanged.
    _dH0_void_NB02 = nb02['lki_void']['dH0_void']       # UV prediction
    _H0_GKI_NB02   = nb02['gki']['H0_GKI']              # analytical H0_GKI
    dH0_void_B = _dH0_void_NB02 * (H0_GKI_B / _H0_GKI_NB02)  # scale to Mode B
    H0_local_B = H0_GKI_B + dH0_void_B
    GKI_val_B = H0_GKI_B - H0_Planck
    LKI_val_B = dH0_void_B

    # ── Tensions ──
    H0_tension_lcdm_B = abs(H0_SH0ES - H0_Planck) / H0_SH0ES_err
    H0_tension_eu_B   = abs(H0_SH0ES - H0_local_B) / H0_SH0ES_err
    S8_tension_lcdm_B = abs(S8_lcdm - S8_obs) / S8_obs_err
    S8_tension_eu_B   = abs(S8_eu_B - S8_obs) / S8_obs_err

    print(f'  H0_GKI(B)    = {H0_GKI_B:.3f}')
    print(f'  H0_local(B)  = {H0_local_B:.3f}')
    print(f'  H0 tension   = {H0_tension_eu_B:.2f}σ (vs {H0_tension_lcdm_B:.1f}σ ΛCDM)')
    print(f'  S8 tension   = {S8_tension_eu_B:.2f}σ (vs {S8_tension_lcdm_B:.1f}σ ΛCDM)')

    # ── Build Mode B results ──
    nb04_B_results = {
        'metadata': {
            'notebook': 'NB04_Observational_Validation',
            'version': 'v1.0',
            'date': str(pd.Timestamp.now()) if 'pd' in dir() else 'N/A',
            'description': 'Observational validation of EU vs LCDM — Mode B (MCMC C2)',
            'upstream': ['NB01_params.json', 'NB02_results.json', 'NB03_B_results.json'],
            'mode': 'B',
            'mode_label': MODE_LABEL,
        },
        'h0_decomposition': {
            'H0_Planck': H0_Planck,
            'H0_GKI': H0_GKI_B,
            'dH0_void': dH0_void_B,
            'H0_local': H0_local_B,
            'H0_SH0ES': H0_SH0ES,
            'H0_SH0ES_err': H0_SH0ES_err,
            'GKI_contribution': GKI_val_B,
            'LKI_contribution': LKI_val_B,
        },
        'bridge': {
            'sigma8_lcdm': sigma8_lcdm,
            'sigma8_eu': sigma8_eu_B,
            'S8_lcdm': S8_lcdm,
            'S8_eu': S8_eu_B,
            'S8_obs': S8_obs,
            'S8_obs_err': S8_obs_err,
            'Omega_m_lcdm': Omega_m_lcdm,
            'Omega_m_eu': Omega_m_eu_B,
            'rd_lcdm': rd_lcdm,
            'rd_eu': rd_eu_B,
        },
        'bao_dr2': nb04_results.get('bao_dr2', {}),
        'shapefit_dr1': nb04_results.get('shapefit_dr1', {}),
        'nogo': nb04_results.get('nogo', {}),
        'age': nb04_results.get('age', {}),
        'grand_chi2': nb04_results.get('grand_chi2', {}),
        'tensions': {
            'H0_SH0ES_sigma_lcdm': H0_tension_lcdm_B,
            'H0_SH0ES_sigma_eu': H0_tension_eu_B,
            'S8_DES_sigma_lcdm': S8_tension_lcdm_B,
            'S8_DES_sigma_eu': S8_tension_eu_B,
        },
        'mode_A_comparison': {
            'H0_tension_eu_A': MODE_A_tensions.get('H0_SH0ES_sigma_eu'),
            'S8_tension_eu_A': MODE_A_tensions.get('S8_DES_sigma_eu'),
            'H0_tension_eu_B': H0_tension_eu_B,
            'S8_tension_eu_B': S8_tension_eu_B,
        },
    }

    # ── Save ──
    output_B = 'NB04_B_results.json'
    with open(output_B, 'w') as f:
        _json.dump(nb04_B_results, f, indent=2, default=float)

    print(f'\n  [SAVED] {output_B}')
    print()
    print('=' * 60)
    print('DUAL MODE COMPLETE')
    print(f'  NB04_A_results.json ✅ (Planck fiducial)')
    print(f'  NB04_B_results.json ✅ (MCMC C2)')
    print('=' * 60)

else:
    print()
    print('§15. MODE B — SKIPPED (C2 not found)')
    print('  Upload NB05_C2_results.json and re-run to get both modes.')


In [ ]:
# ============================================================
# §13.1. COLAB DOWNLOAD
# ============================================================

import zipfile, glob

# Use /content as base (Colab working dir) — robust against os.chdir
_base = '/content'
zip_path = os.path.join(_base, 'NB04_outputs.zip')
_count = 0

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # JSON results — include both A and B
    for _mode in ['A', 'B']:
        _jname = f'NB04_{_mode}_results.json'
        for jpath in [os.path.join(_base, _jname), _jname]:
            if os.path.exists(jpath):
                zf.write(jpath, _jname)
                print(f'  Added: {_jname}')
                _count += 1
                break
        else:
            if _mode == 'A':
                print(f'  [WARN] {_jname} NOT FOUND')
            else:
                print(f'  [INFO] {_jname} not available (Mode A only)')
    
    # Figures — search multiple locations
    _fig_dirs = [os.path.join(_base, 'figures'), 'figures']
    for _fdir in _fig_dirs:
        if os.path.isdir(_fdir):
            for f in sorted(glob.glob(os.path.join(_fdir, '*.png')) + 
                          glob.glob(os.path.join(_fdir, '*.pdf'))):
                arcname = os.path.join('figures', os.path.basename(f))
                zf.write(f, arcname)
                print(f'  Added: {arcname}')
                _count += 1
            break
    else:
        print(f'  [WARN] figures/ directory NOT FOUND')

assert _count > 0, (
    f'FATAL: No files added to zip! '
    f'Check that cells ran successfully. CWD = {os.getcwd()}'
)
print(f'\n[OK] NB04_outputs.zip — {_count} files ({os.path.getsize(zip_path):,} bytes)')

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f'Local env — file at {zip_path}')
